# 🎛️ Biomedical Orchestrator — Unified Query System
## Environment: Python 3.8.6 · ipykernel

**This is the single entry point for ALL queries — both Obj-1 and Obj-2.**

The orchestrator:
1. Detects the **intent** of your query
2. Routes to the correct agent (**KG Agent** or **Structure Agent**)
3. For integrated intents (`FULL_PIPELINE`, `DISEASE_DRUGGABLE_TARGETS`), chains both agents

```
User Query
    │
    ▼
┌─────────────────────────────────────────────────┐
│              ORCHESTRATOR (LangGraph)            │
│                                                  │
│  Node 1: route_query  ← detect intent            │
│      │                                           │
│      ├──── KG intents ──────► Node 2: kg_agent   │  Python 3.8.6 / ipykernel
│      │                            │              │  Uses: ollama pkg, networkx, rapidfuzz
│      │                            ▼              │
│      │                    shared/kg_output.json  │
│      │                                           │
│      ├── Structure intents ► Node 3: struct_agent │  Python 3.10.20 / meeko_env
│      │                            │              │  Uses: meeko, vina, p2rank
│      │                            ▼              │  Called via subprocess
│      │                  shared/structure_output  │
│      │                            .json          │
│      │                                           │
│      └──── Full pipeline ──► Node 2 → Node 3     │
│                                                  │
│  Node 4: synthesize_output                       │
│  Node 5: display_results                         │
└─────────────────────────────────────────────────┘
```

### ⚙️ How cross-environment execution works
The orchestrator runs in `ipykernel` (Python 3.8.6).  
When a Structure intent is detected, it calls `obj_2_structure_agent.ipynb`
via `subprocess` using the `meeko_env` Python interpreter.

### ⚠️ Setup
1. Set `MEEKO_PYTHON` to the path of your meeko_env Python executable
2. Make sure Ollama is running: `ollama serve`
3. Both `shared/` directory and `data/knowledge_graph.gexf` must exist

In [ ]:
# Run once
%pip install langchain langchain-core langgraph rapidfuzz pyvis

## ─── CELL A: Imports & Configuration ───

In [12]:
import os, json, re, subprocess, operator, sys
import networkx as nx
from typing import TypedDict, Annotated, Any, Optional
from IPython.display import display, HTML, Markdown

# LLM
import ollama

# LangChain
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# ─── Rapidfuzz ───
from rapidfuzz import process, fuzz

# ══════════════════════════════════════════════════════════════
# CONFIGURATION — Edit here only
# ══════════════════════════════════════════════════════════════

BASE_DIR = os.getcwd()
SHARED_DIR = os.path.join(BASE_DIR, "shared")
os.makedirs(SHARED_DIR, exist_ok=True)

KG_OUTPUT_FILE        = os.path.join(SHARED_DIR, "kg_output.json")
STRUCTURE_OUTPUT_FILE = os.path.join(SHARED_DIR, "structure_output.json")

# ⚠️ Path to meeko_env Python executable (Windows example below)
# ── UPDATE THIS LINE ──────────────────────────────────────────────────
# To find your path: open Anaconda Prompt → conda activate meeko_env
# → python -c "import sys; print(sys.executable)"
MEEKO_PYTHON = r"C:\Users\Devanshi Di\anaconda3\envs\meeko_env\python.exe"
# ───────────────────────────────────────────────────────────────────────
# Linux/Mac: MEEKO_PYTHON = "/home/user/anaconda3/envs/meeko_env/bin/python"

# Path to obj_2 runner script (auto-generated below)
OBJ2_RUNNER_SCRIPT = os.path.join(BASE_DIR, "run_obj2_agent.py")

LLM_MODEL = "llama3:8b"

print("✅ Orchestrator imports complete  |  Python 3.8.6 / ipykernel")

✅ Orchestrator imports complete  |  Python 3.8.6 / ipykernel


## ─── CELL B: Paste your complete Obj-1 functions here ───

> **Instructions:** Copy all cells from `obj_1_kg_agent.ipynb` Cells B–C into this cell.
> (knowledge_graph load, all def functions, run_pipeline, LangGraph nodes)
> This keeps everything in one kernel.

In [13]:
# =========================
# Core Libraries
# =========================
import os, json, re, requests, operator
import pandas as pd
import numpy as np
import networkx as nx
from typing import TypedDict, Annotated, Any, Optional

# Bioinformatics
from Bio.PDB import PDBParser, PDBIO

# Molecular
from rdkit import Chem
from rdkit.Chem import AllChem

# Subprocess
import subprocess

# Visualization
import py3Dmol
from pyvis.network import Network
from IPython.display import display, HTML

# LLM — Ollama package (available in ipykernel env)
import ollama

# LangChain — NEW additions
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# Shared output directory
SHARED_DIR = os.path.join(os.getcwd(), "shared")
os.makedirs(SHARED_DIR, exist_ok=True)
KG_OUTPUT_FILE = os.path.join(SHARED_DIR, "kg_output.json")

print("✅ Obj-1 imports complete  |  Python 3.8.6 / ipykernel")

# ══════════════════════════════════════════════════════════════
# ORIGINAL OBJ-1 CODE — NOT A SINGLE LINE CHANGED
# Every function is exactly as it appears in your notebook.
# ══════════════════════════════════════════════════════════════

# ─── LLM ───
def call_ollama(model, system, user):
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ],
        options={"temperature": 0}
    )
    return response["message"]["content"]


def safe_json_load(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError("❌ No valid JSON found in LLM output")


def check_step(step_name, value):
    print(f"\n--- {step_name} ---")
    print(value)
    if value is None or value == "" or value == {}:
        raise RuntimeError(f"❌ {step_name} failed")


print("✅ Base utilities loaded")

# ─── KG ───
knowledge_graph = nx.read_gexf("data/knowledge_graph.gexf")
print("KG loaded:", knowledge_graph.number_of_nodes(), "nodes /", knowledge_graph.number_of_edges(), "edges")

# ══════════════════════════════════════════════════════════════
# NATURAL LANGUAGE QUERY NORMALIZER  (NEW)
# ══════════════════════════════════════════════════════════════
# Converts ANY conversational / informal query into a clean
# structured biomedical query BEFORE spelling correction and
# intent detection. This makes the system work for natural queries like:
#   "i want to know which proteins are linked to diabetes"
#   "can you show me the pockets of EGFR"
#   "what drugs could work on alzheimer proteins"
#   "tell me about pathways in lung cancer"
#   "i am curious about targets of metformin"
# ══════════════════════════════════════════════════════════════

def normalize_natural_query(query: str) -> str:
    """
    Step 0 of the pipeline — runs BEFORE spelling correction.
    Converts informal / conversational language into a clean biomedical query.
    Preserves UniProt IDs, SMILES strings, and biomedical entity names exactly.
    """
    system = """You are a biomedical query normalizer.
Your job is to convert informal or conversational user input into a clean, 
direct biomedical query that can be processed by an intent classifier."""

    user = f"""
Convert the following user input into a clean, direct biomedical query.

Rules:
- Keep the same meaning and intent
- Remove conversational filler ("I want to know", "can you tell me", "I am curious", "please show me", "tell me about", "i need to find", "help me understand", etc.)
- Make it sound like a direct question or instruction
- Preserve all biomedical entity names (diseases, proteins, drugs, gene names) EXACTLY as written
- Preserve UniProt IDs (e.g. P07471) unchanged
- Preserve SMILES strings unchanged
- Do NOT add new entities
- Do NOT change disease names
- Do NOT answer the question — only rewrite it
- If already a clean query, return it unchanged

Examples of conversions:
"i want to know which proteins are linked to diabetes" → "What proteins are associated with diabetes?"
"can you show me the binding pockets of EGFR" → "Find binding pockets for EGFR"
"tell me about pathways involved in lung cancer" → "Which pathways are involved in lung cancer?"
"what drugs could possibly work on alzheimer disease proteins" → "What drugs target Alzheimer disease proteins?"
"i am curious about the interaction network of alzheimer" → "Which proteins form the PPI network for Alzheimer disease?"
"help me find drug candidates for the protein P07471" → "Find drug candidates for P07471"
"which are the most important hub proteins for diabetes" → "Which proteins are hub nodes in the diabetes PPI network?"
"can u show me pathways for egfr" → "Which pathways does EGFR participate in?"
"do a full drug discovery for lung cancer" → "Run full drug discovery pipeline for lung cancer"
"i need ligands that bind to p07471" → "Find ligands for P07471"
"show pockets of egfr and find what molecules bind" → "Find binding pockets and ligands for EGFR"
"what are common pathways between diabetes and obesity" → "What are common pathways between diabetes and obesity?"
"find me the hub genes in parkinsons ppi" → "Which proteins are hub nodes in the Parkinson disease PPI network?"
"dock something to egfr" → "Perform docking for EGFR"
"is tp53 druggable" → "Is TP53 druggable?"

User input:
{query}

Return ONLY the normalized query. No explanation. No preamble.
"""
    normalized = call_ollama("llama3:8b", system, user).strip()
    
    # Safety: if LLM returns something too long (hallucinated), fall back to original
    if len(normalized) > len(query) * 3:
        return query
    
    print(f"   🔄 Natural query normalized:")
    print(f"      Input:  {query}")
    print(f"      Output: {normalized}")
    return normalized


print("✅ Natural language query normalizer loaded")
print("   Handles: conversational, informal, casual, abbreviated queries")


# ─── Query Understanding + Intent Detection (UPDATED for natural language) ───

def understand_query(query):
    system = "You are a biomedical query interpreter."
    user = f"""
Explain what the user wants in one sentence.
Do NOT answer the question.
Do NOT add biomedical facts.

Examples:
Q: What are targets for diabetes?
A: The user wants to identify disease-associated proteins.

Q: What proteins bind this molecule?
A: The user wants to identify protein targets of a molecule.

Q: I want to know which proteins are linked to cancer
A: The user wants to identify proteins associated with cancer.

Q: Show me pockets of EGFR
A: The user wants to predict binding pockets for the EGFR protein.

Query:
{query}
"""
    return call_ollama("llama3:8b", system, user)


def optimize_query(query):
    system = "You extract result size constraints from user queries."
    user = f"""
Extract result size constraint and return JSON only.

Rules:
- If user asks for top N → {{ "limit": N }}
- If user asks for all → {{ "limit": "ALL" }}
- If no number mentioned → {{ "limit": 10 }}

Examples:
Q: top 5 targets for diabetes
A: {{ "limit": 5 }}

Q: show all pathways involved in cancer
A: {{ "limit": "ALL" }}

Q: what are targets for Type 2 Diabetes?
A: {{ "limit": 10 }}

Query:
{query}

Return JSON only.
"""
    raw = call_ollama("llama3:8b", system, user)
    print(" Raw optimization output:", raw)
    return safe_json_load(raw)


def detect_intent(query):
    system = "You classify biomedical query intent. You handle both formal and informal/conversational queries."
    user = f"""
Classify the intent of the biomedical query into ONE label from the list below.
The query may be formal, casual, or conversational — classify the MEANING, not the phrasing.

Available intents:

Phase 1 (Knowledge Graph):
- DISEASE_TO_TARGET          → find proteins / genes associated with a disease
- DISEASE_TO_PATHWAY         → find biological pathways associated with a disease
- DRUG_TO_TARGET             → find proteins that a drug acts on
- MOLECULE_TO_TARGET         → find proteins that bind a molecule / SMILES
- TARGET_TO_PATHWAY          → find pathways a protein participates in
- COMMON_DISEASE_DISEASE_PATHWAY  → common pathways between two diseases
- COMMON_DISEASE_PROTEIN_PATHWAY  → common pathways between a disease and a protein
- COMMON_PROTEIN_PROTEIN_PATHWAY  → common pathways between two proteins
- DISEASE_PPI_NETWORK        → protein-protein interaction network for a disease
- DISEASE_PPI_HUBS           → hub / central proteins in a disease PPI network
- MOLECULE_TO_PATHWAY        → pathways affected by a molecule or drug
- MOLECULE_TO_DISEASE        → diseases associated with a molecule or drug target

Phase 2 (Structural / Computational):
- TARGET_STRUCTURE           → get 3D structure of a protein
- TARGET_POCKET              → predict binding pockets of a protein
- TARGET_DRUGGABILITY        → assess if a protein is druggable
- TARGET_LIGAND              → find ligands / small molecules that bind a protein
- TARGET_DOCKING             → perform molecular docking for a protein
- TARGET_VIRTUAL_SCREENING   → virtual screening of compounds against a protein

Phase 3 (Integrated):
- DISEASE_DRUGGABLE_TARGETS  → find druggable protein targets for a disease
- FULL_PIPELINE              → full drug discovery pipeline for a disease
- TARGET_TO_LIGAND_DISCOVERY → discover drug candidates for a specific protein

Examples (covering both formal and natural/conversational):

Q: What are targets for Alzheimer's disease?
A: DISEASE_TO_TARGET

Q: i want to know which proteins are linked to diabetes
A: DISEASE_TO_TARGET

Q: tell me the proteins associated with lung cancer
A: DISEASE_TO_TARGET

Q: which genes are involved in parkinson
A: DISEASE_TO_TARGET

Q: Which pathways are involved in Parkinson's disease?
A: DISEASE_TO_PATHWAY

Q: tell me about pathways in lung cancer
A: DISEASE_TO_PATHWAY

Q: what biological processes are linked to diabetes
A: DISEASE_TO_PATHWAY

Q: What proteins does Metformin act on?
A: DRUG_TO_TARGET

Q: what does aspirin target
A: DRUG_TO_TARGET

Q: which proteins bind to metformin
A: DRUG_TO_TARGET

Q: what proteins bind this molecule O=C1Nc2ccccc2?
A: MOLECULE_TO_TARGET

Q: Which pathways does EGFR participate in?
A: TARGET_TO_PATHWAY

Q: can u show me pathways for egfr
A: TARGET_TO_PATHWAY

Q: what pathways is ACTN1 part of
A: TARGET_TO_PATHWAY

Q: pathways involving P07471
A: TARGET_TO_PATHWAY

Q: Which proteins form the interaction network for Alzheimer disease?
A: DISEASE_PPI_NETWORK

Q: i am curious about the interaction network of alzheimer
A: DISEASE_PPI_NETWORK

Q: show me the ppi network for diabetes
A: DISEASE_PPI_NETWORK

Q: Which proteins are central or hub nodes in the Alzheimer PPI network?
A: DISEASE_PPI_HUBS

Q: which are the most important hub proteins for diabetes
A: DISEASE_PPI_HUBS

Q: find the key hub genes in parkinsons disease network
A: DISEASE_PPI_HUBS

Q: which pathways are affected by metformin?
A: MOLECULE_TO_PATHWAY

Q: which diseases are associated with aspirin targets?
A: MOLECULE_TO_DISEASE

Q: what is the 3D structure of EGFR
A: TARGET_STRUCTURE

Q: show me the structure of P07471
A: TARGET_STRUCTURE

Q: Find binding pockets for EGFR
A: TARGET_POCKET

Q: can you show me the pockets of EGFR
A: TARGET_POCKET

Q: predict binding sites for P07471
A: TARGET_POCKET

Q: where can drugs bind on EGFR
A: TARGET_POCKET

Q: Is TP53 druggable?
A: TARGET_DRUGGABILITY

Q: is egfr a good drug target
A: TARGET_DRUGGABILITY

Q: can we drug P07471
A: TARGET_DRUGGABILITY

Q: Find ligands for P07471
A: TARGET_LIGAND

Q: i need ligands that bind to p07471
A: TARGET_LIGAND

Q: what small molecules bind EGFR
A: TARGET_LIGAND

Q: find molecules that could bind to tp53
A: TARGET_LIGAND

Q: Perform docking for protein P07471
A: TARGET_DOCKING

Q: dock something to egfr
A: TARGET_DOCKING

Q: run molecular docking for P07471
A: TARGET_DOCKING

Q: do virtual screening on EGFR
A: TARGET_VIRTUAL_SCREENING

Q: screen compounds against P07471
A: TARGET_VIRTUAL_SCREENING

Q: Identify druggable targets in Alzheimer's disease
A: DISEASE_DRUGGABLE_TARGETS

Q: what proteins in diabetes could be drug targets
A: DISEASE_DRUGGABLE_TARGETS

Q: find me targetable proteins for lung cancer
A: DISEASE_DRUGGABLE_TARGETS

Q: Run full drug discovery pipeline for lung cancer
A: FULL_PIPELINE

Q: do a full drug discovery for lung cancer
A: FULL_PIPELINE

Q: i want to do complete drug target analysis for alzheimer
A: FULL_PIPELINE

Q: Find drug candidates for EGFR
A: TARGET_TO_LIGAND_DISCOVERY

Q: help me find drug candidates for the protein P07471
A: TARGET_TO_LIGAND_DISCOVERY

Q: suggest drugs that could work on EGFR
A: TARGET_TO_LIGAND_DISCOVERY

Q: what drugs could possibly work on this protein EGFR
A: TARGET_TO_LIGAND_DISCOVERY

Query:
{query}

Return ONLY the intent label. Nothing else.
"""
    return call_ollama("llama3:8b", system, user).strip()


print("✅ Query understanding + intent detection loaded (natural language ready)")


# ─── Entity Extraction ───
UNIPROT_PATTERN = re.compile(r"\b[OPQ][0-9][A-Z0-9]{3}[0-9]\b")
SMILES_PATTERN  = re.compile(r"[A-Za-z0-9@+\-\[\]\(\)=#$]{6,}")

def extract_uniprot(query):
    match = UNIPROT_PATTERN.search(query)
    return match.group() if match else None


def extract_entities(query):
    uniprot_match = UNIPROT_PATTERN.search(query)
    if uniprot_match:
        return {"Disease": [], "Drug": [], "Protein": [uniprot_match.group()], "Molecule": [], "Pathway": []}

    smiles_match = SMILES_PATTERN.search(query)
    if smiles_match and "=" in smiles_match.group():
        return {"Disease": [], "Drug": [], "Protein": [], "Molecule": [smiles_match.group()], "Pathway": []}

    system = "You extract biomedical entities from user queries."
    user = f"""
Extract biomedical entities and return JSON only.

Allowed entity types:
- Disease
- Drug
- Protein
- Molecule
- Pathway

Rules:
- Molecule can be a chemical name OR SMILES string
- Drug names go under Drug
- If unsure between Drug and Molecule → prefer Molecule
- Do NOT invent entities
- Do NOT infer missing entities
- Return empty lists if not present
- Preserve original spelling

Examples:
Q: Predict pockets for EGFR
A: {{"Disease":[],"Drug":[],"Protein":["EGFR"],"Molecule":[],"Pathway":[]}}

Q: Identify druggable targets in Type 2 Diabetes
A: {{"Disease":["Type 2 Diabetes"],"Drug":[],"Protein":[],"Molecule":[],"Pathway":[]}}

Query:
{query}

Return JSON only.
"""
    raw = call_ollama("llama3:8b", system, user)
    print("Raw entity output:", raw)
    return safe_json_load(raw)


def correct_query_spelling_and_entities(query):
    system = "You correct spelling and normalize biomedical entity names."
    user = f"""
Correct spelling mistakes and minor grammar errors.
Return only the corrected query.

Rules:
- Do NOT change the intent
- NO semantic rewriting
- ONLY spelling / grammar
- Preserve the user's meaning
- DO NOT change protein IDs
- Keep SMILES strings unchanged
- DO NOT replace diseases with more specific or different diseases
- DO NOT add or remove words
- DO NOT explain corrections

Examples:
Type 2 diabates → Type 2 diabetes mellitus
Alzeimer disease → Alzheimer disease 18
aplpha 2 plasmin inhibitor deficiency → Alpha-2-plasmin inhibitor deficiency
lung canser → lung cancer
P07471 → P07471

DO NOT DO:
lung cancer → lung carcinoma
breast tumor → breast carcinoma

Query:
{query}
"""
    return call_ollama("llama3:8b", system, user).strip()


print("✅ Entity extraction functions loaded")

# ─── Entity Resolution ───
from rapidfuzz import process, fuzz

def normalize_text_loose(text):
    return re.sub(r'[^a-z0-9]', '', text.lower())

def match_confidence(score):
    if score > 85:   return "high"
    elif score > 65: return "medium"
    return "low"

def normalize_key(text):
    return normalize_text_loose(text) if text else ""

def select_primary_disease(disease_mentions):
    for d in disease_mentions:
        if "deficiency" in d.lower(): return d
    for d in disease_mentions:
        if "mutation" in d.lower(): return d
    return max(disease_mentions, key=len)


def resolve_disease_kg_grounded(query, min_score=40):
    disease_nodes = [
        (node, data.get("disease_name", node))
        for node, data in knowledge_graph.nodes(data=True)
        if data.get("entity_type") == "disease"
    ]
    if not disease_nodes: return None
    disease_names = [name for _, name in disease_nodes]
    match, score, idx = process.extractOne(query, disease_names, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    return {"disease_id": disease_nodes[idx][0], "disease_name": disease_nodes[idx][1],
            "match_score": score, "confidence": "high" if score > 85 else "medium"}


def resolve_protein_kg_grounded(protein_text, min_score=40):
    query_raw  = protein_text.strip()
    query_norm = normalize_text_loose(protein_text)
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "protein":
            if (node.lower() == query_raw.lower() or
                data.get("protein_name", "").lower() == query_raw.lower() or
                data.get("gene_name", "").lower() == query_raw.lower()):
                return {"protein_id": node, "protein_name": data.get("protein_name", node),
                        "gene_name": data.get("gene_name", ""), "match_score": 100, "confidence": "high"}
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "protein":
            combined = f"{node} {data.get('protein_name','')} {data.get('gene_name','')}"
            if normalize_text_loose(combined) == query_norm:
                return {"protein_id": node, "protein_name": data.get("protein_name", node),
                        "gene_name": data.get("gene_name", ""), "match_score": 95, "confidence": "high"}
    candidates, texts = [], []
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "protein":
            combined = f"{node} {data.get('protein_name','')} {data.get('gene_name','')}"
            candidates.append((node, data)); texts.append(normalize_text_loose(combined))
    if not texts: return None
    match, score, idx = process.extractOne(query_norm, texts, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    node_id, node_data = candidates[idx]
    return {"protein_id": node_id, "protein_name": node_data.get("protein_name", node_id),
            "gene_name": node_data.get("gene_name", ""), "match_score": score, "confidence": match_confidence(score)}


def resolve_pathway_kg_grounded(query, min_score=40):
    if not query: return None
    query_norm = query.lower().strip()
    pathway_nodes = [(node, data.get("pathway_name", "").lower())
                     for node, data in knowledge_graph.nodes(data=True) if data.get("entity_type") == "pathway"]
    if not pathway_nodes: return None
    choices = [name for _, name in pathway_nodes]
    match, score, idx = process.extractOne(query_norm, choices, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    matched_node = pathway_nodes[idx][0]
    node_data = knowledge_graph.nodes[matched_node]
    return {"pathway_id": matched_node, "pathway_name": node_data.get("pathway_name", ""),
            "source": node_data.get("source", ""), "match_score": score}


def resolve_drug_kg_grounded(drug_text, min_score=40):
    drug_text = drug_text.lower()
    best_match, best_score = None, 0
    for node_id, node_data in knowledge_graph.nodes(data=True):
        drug_name = node_data.get("drug_name") or node_data.get("name")
        if not drug_name: continue
        score = fuzz.partial_ratio(drug_text, drug_name.lower())
        if score > best_score: best_score = score; best_match = (node_id, node_data)
    if best_match and best_score >= min_score:
        node_id, node_data = best_match
        return {"drug_id": node_id, "drug_name": node_data.get("drug_name") or node_data.get("name"),
                "match_score": best_score, "confidence": match_confidence(best_score)}
    return None


def resolve_molecule_kg_grounded(molecule_text, min_score=60):
    query_norm = normalize_text_loose(molecule_text)
    candidates, texts = [], []
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "molecule":
            name   = data.get("molecule_name", "")
            smiles = data.get("smiles", "")
            label  = data.get("label", "")
            candidates.append((node, data)); texts.append(normalize_text_loose(f"{name} {smiles} {label}"))
    if not texts: return None
    match, score, idx = process.extractOne(query_norm, texts, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    node_id, node_data = candidates[idx]
    return {"molecule_id": node_id, "molecule_name": node_data.get("molecule_name", node_id),
            "smiles": node_data.get("smiles", ""), "match_score": score, "confidence": match_confidence(score)}


def normalize_entities_kg_grounded(entities):
    normalized = {"Disease": [], "Protein": [], "Drug": [], "Molecule": [], "Pathway": []}
    if entities.get("Disease"):
        resolved = resolve_disease_kg_grounded(entities["Disease"][0])
        if resolved: normalized["Disease"].append(resolved)
    if entities.get("Protein"):
        resolved = resolve_protein_kg_grounded(entities["Protein"][0])
        if resolved: normalized["Protein"].append(resolved)
    if entities.get("Drug"):
        resolved_drug = resolve_drug_kg_grounded(entities["Drug"][0])
        if resolved_drug: normalized["Drug"].append(resolved_drug)
        else:
            resolved_mol = resolve_molecule_kg_grounded(entities["Drug"][0])
            if resolved_mol: normalized["Molecule"].append(resolved_mol)
    if entities.get("Molecule") and not normalized["Drug"]:
        resolved = resolve_molecule_kg_grounded(entities["Molecule"][0])
        if resolved: normalized["Molecule"].append(resolved)
    if entities.get("Pathway"):
        resolved = resolve_pathway_kg_grounded(entities["Pathway"][0])
        if resolved: normalized["Pathway"].append(resolved)
    return normalized


print("✅ Entity resolution functions loaded")

# ─── KG Query Functions ───

def build_disease_ppi_subgraph(disease_id, max_nodes=300):
    G_sub = nx.Graph()
    if disease_id not in knowledge_graph: return G_sub
    disease_proteins = set(nbr for nbr in knowledge_graph.neighbors(disease_id)
                           if knowledge_graph.nodes[nbr].get("entity_type") == "protein")
    if not disease_proteins: return G_sub
    expanded_proteins = set(disease_proteins)
    for p in disease_proteins:
        for nbr in knowledge_graph.neighbors(p):
            if knowledge_graph.nodes[nbr].get("entity_type") == "protein":
                expanded_proteins.add(nbr)
                for nbr2 in knowledge_graph.neighbors(nbr):
                    if knowledge_graph.nodes[nbr2].get("entity_type") == "protein":
                        expanded_proteins.add(nbr2)
    expanded_proteins = list(expanded_proteins)[:max_nodes]
    for p in expanded_proteins:
        G_sub.add_node(p, label=knowledge_graph.nodes[p].get("protein_name", p))
    VALID_PPI = ["interacts_with", "binds", "ppi"]
    for u, v, data in knowledge_graph.edges(data=True):
        if data.get("relation") in VALID_PPI and u in expanded_proteins and v in expanded_proteins:
            G_sub.add_edge(u, v)
    return G_sub


def kg_disease_to_targets_direct(resolved_disease, limit=None, ranking=None):
    disease_id = resolved_disease["disease_id"]
    results = []
    for nbr in knowledge_graph.neighbors(disease_id):
        edge = knowledge_graph[disease_id][nbr]
        node = knowledge_graph.nodes[nbr]
        if node.get("entity_type") == "protein":
            results.append({"protein_id": nbr, "protein_name": node.get("protein_name", nbr),
                            "gene_name": node.get("gene_name", ""), "relation": edge.get("relation")})
    if ranking and results:
        G = build_disease_ppi_subgraph(disease_id)
        if G.number_of_nodes() > 0:
            scores = nx.degree_centrality(G) if ranking == "degree" else nx.pagerank(G) if ranking == "pagerank" else {}
            for r in results: r["score"] = round(scores.get(r["protein_id"], 0), 4)
            results = sorted(results, key=lambda x: x.get("score", 0), reverse=True)
    if isinstance(limit, int): results = results[:limit]
    return {"disease": resolved_disease, "targets": results, "ranking_method": ranking if ranking else "none"}


def kg_disease_to_pathways(resolved_disease, limit=None):
    disease_id = resolved_disease["disease_id"]
    pathways = {}
    for protein in knowledge_graph.neighbors(disease_id):
        if knowledge_graph.nodes[protein].get("entity_type") != "protein": continue
        for path in knowledge_graph.neighbors(protein):
            node = knowledge_graph.nodes[path]
            if node.get("entity_type") == "pathway":
                pathways[path] = {"pathway_id": path, "pathway_name": node.get("pathway_name", path),
                                  "source": node.get("source", "")}
    results = list(pathways.values())
    if limit not in [None, "ALL"]: results = results[:int(limit)]
    return {"disease": resolved_disease, "pathways": results}


def kg_target_to_pathways(protein_id, limit=None):
    results = []
    for nbr in knowledge_graph.neighbors(protein_id):
        node = knowledge_graph.nodes[nbr]
        if node.get("entity_type") == "pathway":
            results.append({"pathway_id": nbr, "pathway_name": node.get("pathway_name", nbr),
                            "source": node.get("source", "")})
    if limit not in [None, "ALL"]: results = results[:int(limit)]
    return results


def kg_disease_ppi_network(disease):
    disease_id = disease["disease_id"]
    disease_proteins = set(nbr for nbr in knowledge_graph.neighbors(disease_id)
                           if knowledge_graph.nodes[nbr].get("entity_type") == "protein")
    if not disease_proteins: return {"proteins": [], "edges": []}
    ppi_edges, all_proteins = [], set(disease_proteins)
    for u, v, data in knowledge_graph.edges(data=True):
        if data.get("relation") == "interacts_with" and (u in disease_proteins or v in disease_proteins):
            ppi_edges.append((u, v))
            for n in [u, v]:
                if knowledge_graph.nodes[n].get("entity_type") == "protein": all_proteins.add(n)
    return {"proteins": list(all_proteins), "edges": ppi_edges}


def kg_disease_ppi_hubs(disease, top_k=10):
    G_sub = build_disease_ppi_subgraph(disease["disease_id"])
    if G_sub.number_of_nodes() == 0 or G_sub.number_of_edges() == 0:
        return [{"message": "No PPI data available for this disease"}]
    degree = nx.degree_centrality(G_sub)
    hubs   = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"protein_id": node, "score": score} for node, score in hubs]


def kg_drug_to_targets(drug_id, limit=None):
    targets = []
    if drug_id not in knowledge_graph: return targets
    for nbr in knowledge_graph.neighbors(drug_id):
        edge = knowledge_graph[drug_id][nbr]
        if edge.get("relation") == "targets":
            pdata = knowledge_graph.nodes[nbr]
            targets.append({"protein_id": nbr, "protein_name": pdata.get("protein_name", nbr),
                            "gene_name": pdata.get("gene_name", ""),
                            "affinity_value": edge.get("affinity_value"),
                            "confidence_score": edge.get("confidence_score")})
    if limit not in [None, "ALL"]: targets = targets[:int(limit)]
    return targets


def kg_molecule_to_targets(molecule_node_id, limit=None):
    targets = {}
    if molecule_node_id not in knowledge_graph: return []
    for nbr in knowledge_graph.neighbors(molecule_node_id):
        edge = knowledge_graph[molecule_node_id][nbr]
        if edge.get("relation") == "has_structure":
            drug = nbr
            for prot in knowledge_graph.neighbors(drug):
                edge2 = knowledge_graph[drug][prot]
                if edge2.get("relation") == "targets":
                    pdata = knowledge_graph.nodes[prot]
                    targets[prot] = {"protein_id": prot, "protein_name": pdata.get("protein_name", prot),
                                     "gene_name": pdata.get("gene_name", ""), "drug_id": drug}
    results = list(targets.values())
    if limit not in [None, "ALL"]: results = results[:int(limit)]
    return results


def compute_network_scores(G):
    return nx.degree_centrality(G), nx.pagerank(G)


def compute_pathway_score(protein_id):
    return len(kg_target_to_pathways(protein_id))


def compute_proximity(G, disease_proteins):
    proximity = {}
    for node in G.nodes():
        distances = []
        for dp in disease_proteins:
            try: distances.append(nx.shortest_path_length(G, node, dp))
            except: continue
        proximity[node] = sum(1/(1+nx.shortest_path_length(G,node,dp)) for dp in disease_proteins if nx.has_path(G,node,dp)) if distances else 0
    return proximity


def prioritize_targets_all_methods(disease, top_k=10):
    disease_id = disease["disease_id"]
    targets    = kg_disease_to_targets_direct(disease)["targets"]
    G          = build_disease_ppi_subgraph(disease_id)
    disease_proteins = set(nbr for nbr in knowledge_graph.neighbors(disease_id)
                           if knowledge_graph.nodes[nbr].get("entity_type") == "protein")
    degree_scores   = nx.degree_centrality(G)
    pagerank_scores = nx.pagerank(G)
    proximity = {}
    for node in G.nodes():
        distances = []
        for dp in disease_proteins:
            try: distances.append(nx.shortest_path_length(G, node, dp))
            except: continue
        proximity[node] = 1/(1+min(distances)) if distances else 0
    pathway_scores = {t["protein_id"]: len(kg_target_to_pathways(t["protein_id"])) for t in targets}
    return {
        "degree":    sorted(targets, key=lambda x: degree_scores.get(x["protein_id"],0),   reverse=True)[:top_k],
        "pagerank":  sorted(targets, key=lambda x: pagerank_scores.get(x["protein_id"],0), reverse=True)[:top_k],
        "proximity": sorted(targets, key=lambda x: proximity.get(x["protein_id"],0),       reverse=True)[:top_k],
        "pathway":   sorted(targets, key=lambda x: pathway_scores.get(x["protein_id"],0),  reverse=True)[:top_k],
    }


print("✅ KG query functions loaded")

# ─── Visualization & Explanation ───

def plot_ppi_network_interactive(G_ppi, disease_id, top_targets=None, notebook=False, hub_percent=0.2):
    net = Network(height="750px", width="100%", bgcolor="#ffffff", font_color="black",
                  notebook=notebook, cdn_resources="in_line")
    net.force_atlas_2based(); net.toggle_physics(True)
    degree     = dict(G_ppi.degree())
    centrality = nx.degree_centrality(G_ppi)
    top_targets = set(top_targets) if top_targets else set()
    drug_targets = set()
    for node in G_ppi.nodes():
        if node in knowledge_graph:
            for nbr in knowledge_graph.neighbors(node):
                if knowledge_graph[nbr][node].get("relation") == "targets": drug_targets.add(node)
    sorted_deg = sorted(degree.values(), reverse=True)
    hub_threshold = sorted_deg[max(1, int(hub_percent*len(sorted_deg)))] if degree else 0
    for node in G_ppi.nodes():
        protein_name = knowledge_graph.nodes[node].get("protein_name", node)
        role = G_ppi.nodes[node].get("role", "neighbor_protein")
        if node in top_targets:            color = "#ff0000"
        elif node in drug_targets:         color = "#e31a1c"
        elif role == "disease_protein":    color = "#33a02c"
        elif degree.get(node,0) >= hub_threshold: color = "#ff7f00"
        else:                              color = "#1f78b4"
        size  = 15 + centrality.get(node, 0) * 120
        title = f"Protein: {protein_name}\nRole: {role}\nDegree: {degree.get(node,0)}\nCentrality: {centrality.get(node,0):.4f}\nDrug Target: {'Yes' if node in drug_targets else 'No'}"
        net.add_node(node, label=protein_name, color=color, size=size, title=title)
    for u, v in G_ppi.edges():
        net.add_edge(u, v, value=G_ppi[u][v].get("weight", 1), color="#999999")
    return net


def format_targets_with_ranking(results):
    if not isinstance(results, dict): return "\n❌ Invalid results format.\n"
    if "disease" not in results and "targets" not in results: return "\n❌ No disease-target data found.\n"
    disease_name = results.get("disease", {}).get("disease_name", "Unknown Disease")
    text = f"\n🧬 Targets associated with {disease_name}:\n\n"
    targets = results.get("targets", [])
    if not targets: return text + "No targets found.\n"
    for i, t in enumerate(targets, 1):
        gene = t.get("gene_name") or t.get("protein_name") or "Unknown"
        text += f"{i}. {gene} (score: {t['score']})\n" if "score" in t else f"{i}. {gene}\n"
    return text


def format_results_plain_text(results, intent):
    if intent == "DISEASE_TO_TARGET":
        disease = results["disease"]["disease_name"]
        text = f"\n🧬 Targets associated with {disease}:\n\n"
        for i, t in enumerate(results.get("all_targets", []), 1): text += f"{i}. {t['gene_name']}\n"
        text += "\n\n🔬 Target Prioritization:\n"
        text += format_targets_with_ranking(results.get("ranked_targets", {}))
        return text
    elif intent == "DISEASE_TO_PATHWAY":
        disease = results["disease"]["disease_name"]
        text = f"\n🧪 Pathways involved in {disease}:\n\n"
        for i, p in enumerate(results.get("pathways", []), 1): text += f"{i}. {p['pathway_name']}\n"
        return text
    elif intent == "TARGET_TO_PATHWAY":
        protein = results.get("protein", {}).get("gene_name", "")
        text = f"\n🧬 Pathways involving {protein}:\n\n"
        for i, p in enumerate(results.get("pathways", []), 1): text += f"{i}. {p['pathway_name']}\n"
        return text
    elif intent == "DISEASE_PPI_NETWORK":
        disease = results["disease"]["disease_name"]
        summary = results.get("ppi_network_summary", {})
        return f"\n🔗 PPI Network for {disease}:\n\nProteins: {summary.get('num_proteins',0)}\nInteractions: {summary.get('num_interactions',0)}\n"
    elif intent == "DISEASE_PPI_HUBS":
        disease = results["disease"]["disease_name"]
        text = f"\n⭐ Hub proteins in {disease}:\n\n"
        for i, p in enumerate(results.get("hub_proteins", []), 1):
            gene = p.get("gene_name", p.get("protein_id", "Unknown"))
            degree = p.get("degree")
            text += f"{i}. {gene} (degree: {degree})\n" if degree is not None else f"{i}. {gene}\n"
        return text
    else:
        return str(results)


def explain_results_with_llm(query, results, intent):
    system = "You are a biomedical research assistant explaining computational biology results."
    common_instruction = """
Instructions:
- Explain ONLY using the provided results (no generic biology)
- For each important entity, explain its role using given data
- If pathway functions are provided, use them explicitly
- If ranking is provided, explain WHY top items are ranked higher
- Identify biological patterns (e.g., MAPK signaling cluster, insulin signaling)

Output:
- 6–8 lines explanation
- Mention specific targets/pathways
- Include reasoning (NOT description)
"""
    if intent == "DISEASE_TO_TARGET":
        targets = [t["gene_name"] for t in results.get("all_targets", [])]
        ranked  = results.get("ranked_targets", {})
        user = f"Query: {query}\n\nDisease: {results['disease']['disease_name']}\n\nTargets:\n{targets}\n\nRanking:\n{ranked}\n\n{common_instruction}"
    elif intent == "DISEASE_TO_PATHWAY":
        pathways = [p["pathway_name"] for p in results.get("pathways", [])]
        user = f"Query: {query}\n\nDisease: {results['disease']['disease_name']}\n\nPathways:\n{pathways}\n\n{common_instruction}"
    elif intent == "TARGET_TO_PATHWAY":
        protein  = results.get("protein", {}).get("gene_name", "")
        pathways = [p["pathway_name"] for p in results.get("pathways", [])]
        user = f"Query: {query}\n\nProtein: {protein}\n\nPathways:\n{pathways}\n\n{common_instruction}"
    elif intent == "DISEASE_PPI_NETWORK":
        user = f"Query: {query}\n\nNetwork summary:\n{results.get('ppi_network_summary',{})}\n\nProteins:\n{results.get('ppi_network_proteins',[])}\n\n{common_instruction}"
    elif intent == "DISEASE_PPI_HUBS":
        user = f"Query: {query}\n\nHub proteins:\n{results.get('hub_proteins',[])}\n\n{common_instruction}"
    else:
        user = f"Query: {query}\n\nResults:\n{results}\n\n{common_instruction}"
    return call_ollama("llama3:8b", system, user)


print("✅ Visualization & explanation functions loaded")

# ─── Original run_pipeline() — COMPLETELY UNCHANGED ───

def run_pipeline(query):

    print("\n========== PIPELINE START ==========\n")

    # 0a. Natural language normalization
    query = normalize_natural_query(query)
    print("Normalized Query:", query)

    # 0b. Spelling correction
    query = correct_query_spelling_and_entities(query)
    print("Corrected Query:", query)

    # 1. Query understanding
    print("\n--- Query Understanding ---")
    print(understand_query(query))

    # 2. Intent
    intent = detect_intent(query)
    print("\n--- Intent ---"); print(intent)

    # 3. Entity extraction
    entities = extract_entities(query)
    print("\n--- Entities ---"); print(entities)

    # 4. Normalize
    normalized = normalize_entities_kg_grounded(entities)
    normalized["Protein"] = [p for p in normalized.get("Protein", [])
                              if p.get("confidence") == "high" or p.get("match_score", 0) >= 80]
    print("\n--- Normalized ---"); print(normalized)

    # 5. Query optimization
    constraints = optimize_query(query)
    limit = constraints.get("limit", 10)
    match = re.search(r'\btop\s*(\d+)', query.lower())
    if match: limit = int(match.group(1))
    print("\nTop results:", limit)

    results = {}

    if intent == "DISEASE_TO_TARGET" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        raw_targets    = kg_disease_to_targets_direct(disease, limit="ALL")
        ranked_targets = prioritize_targets_all_methods(disease, top_k=10)
        results = {"disease": disease, "all_targets": raw_targets["targets"], "ranked_targets": ranked_targets}

    elif intent == "DISEASE_TO_PATHWAY" and normalized.get("Disease"):
        results = kg_disease_to_pathways(normalized["Disease"][0], limit)

    elif intent == "TARGET_TO_PATHWAY" and normalized.get("Protein"):
        protein = normalized["Protein"][0]
        results = {"protein": protein, "pathways": kg_target_to_pathways(protein["protein_id"], limit)}

    elif intent == "DISEASE_PPI_NETWORK" and normalized.get("Disease"):
        disease    = normalized["Disease"][0]
        disease_id = disease["disease_id"]
        G_ppi      = build_disease_ppi_subgraph(disease_id)
        if G_ppi.number_of_nodes() > 0:
            print(f"\nRendering simple PPI network for {disease['disease_name']} ...")
            net  = plot_ppi_network_interactive(G_ppi, disease_id=disease_id, notebook=True)
            html = net.generate_html()
            display(HTML(html))
            html_file = f"ppi_{disease_id}.html"
            with open(html_file, "w", encoding="utf-8") as f: f.write(html)
            results = {"disease": disease,
                       "ppi_network_summary": {"num_proteins": G_ppi.number_of_nodes(), "num_interactions": G_ppi.number_of_edges()},
                       "ppi_network_proteins": list(G_ppi.nodes())}
        else:
            print("No PPI interactions available.")
            results = {"disease": disease, "message": "No PPI interactions found"}

    elif intent == "DISEASE_PPI_HUBS" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        results = {"disease": disease, "hub_proteins": kg_disease_ppi_hubs(disease, limit)}

    else:
        results = {"message": "No valid result found."}

    formatted_output = format_results_plain_text(results, intent)
    print("\n--- Results ---"); print(formatted_output)

    explanation = explain_results_with_llm(query, results, intent)
    print("\n--- Explanation ---"); print(explanation)
    print("\n=========== PIPELINE END ===========\n")

    return results, intent, explanation


print("✅ Original run_pipeline() loaded")

# ══════════════════════════════════════════════════════════════
# LANGGRAPH STATE
# ══════════════════════════════════════════════════════════════

class KGState(TypedDict):
    query:            str
    corrected_query:  str
    intent:           str
    entities:         dict
    normalized:       dict
    limit:            Any
    kg_results:       dict
    formatted_text:   str
    explanation:      str
    druggable_targets: list   # top proteins → passed to Obj-2
    messages:         Annotated[list, operator.add]


# ─── Node 1: Preprocess ───
def node_preprocess(state: KGState) -> KGState:
    query      = state["query"]
    # Step 0: normalize natural language before spelling correction
    query      = normalize_natural_query(query)
    corrected  = correct_query_spelling_and_entities(query)
    intent     = detect_intent(corrected)
    entities   = extract_entities(corrected)
    normalized = normalize_entities_kg_grounded(entities)
    normalized["Protein"] = [p for p in normalized.get("Protein", [])
                              if p.get("confidence") == "high" or p.get("match_score", 0) >= 80]
    constraints = optimize_query(corrected)
    limit       = constraints.get("limit", 10)
    m = re.search(r'\btop\s*(\d+)', corrected.lower())
    if m: limit = int(m.group(1))
    understanding = understand_query(corrected)
    print(f"\n🔍 Preprocessed | Intent: {intent} | Understanding: {understanding}")
    return {**state, "corrected_query": corrected, "intent": intent, "entities": entities,
            "normalized": normalized, "limit": limit,
            "messages": [AIMessage(content=f"Intent: {intent}. Understanding: {understanding}")]}


# ─── Node 2: KG Query — mirrors run_pipeline() routing exactly ───
def node_kg_query(state: KGState) -> KGState:
    intent     = state["intent"]
    normalized = state["normalized"]
    limit      = state["limit"]
    results    = {}

    print(f"\n🧬 KG Query | Intent: {intent}")

    if intent == "DISEASE_TO_TARGET" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        raw     = kg_disease_to_targets_direct(disease, limit="ALL")
        ranked  = prioritize_targets_all_methods(disease, top_k=10)
        results = {"disease": disease, "all_targets": raw["targets"], "ranked_targets": ranked}

    elif intent == "DISEASE_TO_PATHWAY" and normalized.get("Disease"):
        results = kg_disease_to_pathways(normalized["Disease"][0], limit)

    elif intent == "TARGET_TO_PATHWAY" and normalized.get("Protein"):
        protein = normalized["Protein"][0]
        results = {"protein": protein, "pathways": kg_target_to_pathways(protein["protein_id"], limit)}

    elif intent == "DRUG_TO_TARGET" and normalized.get("Drug"):
        drug    = normalized["Drug"][0]
        targets = kg_drug_to_targets(drug["drug_id"], limit)
        results = {"drug": drug, "targets": targets}

    elif intent == "MOLECULE_TO_TARGET" and normalized.get("Molecule"):
        mol     = normalized["Molecule"][0]
        targets = kg_molecule_to_targets(mol["molecule_id"], limit)
        results = {"molecule": mol, "targets": targets}

    elif intent == "DISEASE_PPI_NETWORK" and normalized.get("Disease"):
        disease    = normalized["Disease"][0]
        disease_id = disease["disease_id"]
        G_ppi      = build_disease_ppi_subgraph(disease_id)
        if G_ppi.number_of_nodes() > 0:
            net  = plot_ppi_network_interactive(G_ppi, disease_id=disease_id, notebook=True)
            html = net.generate_html()
            display(HTML(html))
            with open(f"ppi_{disease_id}.html", "w", encoding="utf-8") as f: f.write(html)
            results = {"disease": disease,
                       "ppi_network_summary": {"num_proteins": G_ppi.number_of_nodes(), "num_interactions": G_ppi.number_of_edges()},
                       "ppi_network_proteins": list(G_ppi.nodes())}
        else:
            results = {"disease": disease, "message": "No PPI interactions found"}

    elif intent == "DISEASE_PPI_HUBS" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        results = {"disease": disease, "hub_proteins": kg_disease_ppi_hubs(disease, int(limit) if isinstance(limit, int) else 10)}

    elif intent in ["DISEASE_DRUGGABLE_TARGETS", "FULL_PIPELINE"] and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        raw     = kg_disease_to_targets_direct(disease, limit="ALL")
        ranked  = prioritize_targets_all_methods(disease, top_k=10)
        results = {"disease": disease, "all_targets": raw["targets"], "ranked_targets": ranked}

    else:
        results = {"message": "No valid result found. Check entity extraction."}

    formatted = format_results_plain_text(results, intent)
    print(formatted)
    return {**state, "kg_results": results, "formatted_text": formatted,
            "messages": [AIMessage(content=formatted)]}


# ─── Node 3: LLM Explain ───
def node_llm_explain(state: KGState) -> KGState:
    explanation = explain_results_with_llm(state["corrected_query"], state["kg_results"], state["intent"])
    print("\n--- LLM Explanation ---"); print(explanation)
    return {**state, "explanation": explanation, "messages": [AIMessage(content=explanation)]}


# ─── Node 4: Export shared ───
def node_export_shared(state: KGState) -> KGState:
    """
    Write shared/kg_output.json so Obj-2 knows which UniProt IDs to structurally analyse.
    """
    results = state["kg_results"]
    intent  = state["intent"]

    druggable_targets = []
    if intent in ["DISEASE_TO_TARGET", "DISEASE_DRUGGABLE_TARGETS", "FULL_PIPELINE"]:
        ranked = results.get("ranked_targets", {})
        top_pr = ranked.get("pagerank", [])[:3]
        druggable_targets = [{"protein_id": t["protein_id"], "gene_name": t.get("gene_name", ""),
                              "protein_name": t.get("protein_name", "")} for t in top_pr]

    shared_payload = {
        "query":             state["corrected_query"],
        "intent":            intent,
        "disease":           results.get("disease", {}),
        "druggable_targets": druggable_targets,
        "kg_summary":        state["formatted_text"],
        "llm_explanation":   state["explanation"]
    }
    with open(KG_OUTPUT_FILE, "w") as f:
        json.dump(shared_payload, f, indent=2)
    print(f"\n✅ shared/kg_output.json written | Druggable targets: {[t['gene_name'] for t in druggable_targets]}")
    return {**state, "druggable_targets": druggable_targets,
            "messages": [AIMessage(content=f"KG output exported. Targets for Obj-2: {druggable_targets}")]}


print("✅ LangGraph nodes defined")

# ══════════════════════════════════════════════════════════════
# BUILD LANGGRAPH
# Flow: preprocess → kg_query → llm_explain → export_shared → END
# ══════════════════════════════════════════════════════════════

kg_builder = StateGraph(KGState)
kg_builder.add_node("preprocess",    node_preprocess)
kg_builder.add_node("kg_query",      node_kg_query)
kg_builder.add_node("llm_explain",   node_llm_explain)
kg_builder.add_node("export_shared", node_export_shared)

kg_builder.set_entry_point("preprocess")
kg_builder.add_edge("preprocess",    "kg_query")
kg_builder.add_edge("kg_query",      "llm_explain")
kg_builder.add_edge("llm_explain",   "export_shared")
kg_builder.add_edge("export_shared", END)

kg_agent = kg_builder.compile()
print("✅ Obj-1 KG LangGraph compiled:", list(kg_builder.nodes.keys()))

def run_kg_agent(query: str) -> dict:
    """
    Run the Obj-1 KG Agent through LangGraph.
    Writes shared/kg_output.json for the Orchestrator and Obj-2.
    """
    print("\n" + "═"*60)
    print(f"  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel")
    print(f"  Query: {query}")
    print("═"*60)

    initial: KGState = {
        "query": query, "corrected_query": "", "intent": "", "entities": {},
        "normalized": {}, "limit": 10, "kg_results": {}, "formatted_text": "",
        "explanation": "", "druggable_targets": [],
        "messages": [HumanMessage(content=query)]
    }
    final = kg_agent.invoke(initial)
    print("\n" + "═"*60)
    print("  OBJ-1 COMPLETE — shared/kg_output.json ready")
    print("═"*60)
    return final


print("✅ run_kg_agent() ready")

✅ Obj-1 imports complete  |  Python 3.8.6 / ipykernel
✅ Base utilities loaded
KG loaded: 80756 nodes / 241886 edges
✅ Natural language query normalizer loaded
   Handles: conversational, informal, casual, abbreviated queries
✅ Query understanding + intent detection loaded (natural language ready)
✅ Entity extraction functions loaded
✅ Entity resolution functions loaded
✅ KG query functions loaded
✅ Visualization & explanation functions loaded
✅ Original run_pipeline() loaded
✅ LangGraph nodes defined
✅ Obj-1 KG LangGraph compiled: ['preprocess', 'kg_query', 'llm_explain', 'export_shared']
✅ run_kg_agent() ready


## ─── CELL C: Obj-2 Runner Script (auto-generated) ───

In [14]:
# ══════════════════════════════════════════════════════════════
# CELL C: Verify run_obj2_agent.py exists
# ══════════════════════════════════════════════════════════════
# run_obj2_agent.py is already fully built — no generation needed.
# Just confirm it is in the same folder as this notebook.

import os

if os.path.exists(OBJ2_RUNNER_SCRIPT):
    print(f"✅ run_obj2_agent.py found at: {OBJ2_RUNNER_SCRIPT}")
else:
    print(f"❌ run_obj2_agent.py NOT found at: {OBJ2_RUNNER_SCRIPT}")
    print("   Make sure run_obj2_agent.py is in the same folder as this notebook.")
    print(f"   Current folder: {os.getcwd()}")


✅ run_obj2_agent.py found at: C:\Dissertation\code\run_obj2_agent.py


## ─── CELL D: Intent Classification ───

In [15]:
# ══════════════════════════════════════════════════════════════
# INTENT SETS — decides which agent(s) to route to
# ══════════════════════════════════════════════════════════════

KG_ONLY_INTENTS = {
    "DISEASE_TO_TARGET",
    "DISEASE_TO_PATHWAY",
    "DRUG_TO_TARGET",
    "MOLECULE_TO_TARGET",
    "TARGET_TO_PATHWAY",
    "COMMON_DISEASE_DISEASE_PATHWAY",
    "COMMON_DISEASE_PROTEIN_PATHWAY",
    "COMMON_PROTEIN_PROTEIN_PATHWAY",
    "DISEASE_PPI_NETWORK",
    "DISEASE_PPI_HUBS",
    "MOLECULE_TO_PATHWAY",
    "MOLECULE_TO_DISEASE",
}

STRUCTURE_ONLY_INTENTS = {
    "TARGET_STRUCTURE",
    "TARGET_POCKET",
    "TARGET_DRUGGABILITY",
    "TARGET_LIGAND",
    "TARGET_DOCKING",
    "TARGET_VIRTUAL_SCREENING",
    "TARGET_TO_LIGAND_DISCOVERY",
}

FULL_PIPELINE_INTENTS = {
    "DISEASE_DRUGGABLE_TARGETS",
    "FULL_PIPELINE",
}


def classify_route(intent: str) -> str:
    """Returns 'kg', 'structure', or 'full'."""
    if intent in KG_ONLY_INTENTS:       return "kg"
    if intent in STRUCTURE_ONLY_INTENTS: return "structure"
    if intent in FULL_PIPELINE_INTENTS:  return "full"
    return "kg"   # safe default


print("✅ Intent classification loaded")
print(f"   KG intents:         {len(KG_ONLY_INTENTS)}")
print(f"   Structure intents:  {len(STRUCTURE_ONLY_INTENTS)}")
print(f"   Full pipeline:      {len(FULL_PIPELINE_INTENTS)}")

✅ Intent classification loaded
   KG intents:         12
   Structure intents:  7
   Full pipeline:      2


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# MISSING FUNCTIONS FROM OBJ-1
#
# ROOT CAUSE OF WRONG ANSWERS:
# 1. prioritize_targets_all_methods() in orchestrator was a stub — it called
#    get_structural_druggability_score() and get_pubmed_literature_score()
#    which DID NOT EXIST in orchestrator kernel. So ranked_targets was always
#    empty → "No disease-target data found."
#
# 2. The orchestrator called prioritize_targets_all_methods(disease, top_k=10)
#    but obj-1 signature is (disease, targets=None, top_k=10). Without passing
#    targets= the function re-fetches from KG internally — fine — but scoring
#    still failed because the API functions were missing.
#
# 3. format_results_plain_text() in orchestrator was stripped — missing branches
#    for PATHWAY_LITERATURE_VALIDATION, CROSS_DISEASE_PATHWAYS,
#    PATHWAY_HUB_ANALYSIS, PATHWAY_TO_DISEASES, PATHWAY_DISEASE_BURDEN.
#
# 4. explain_results_with_llm() was similarly stripped.
#
# FIX: Copy exact functions from obj_1_kg_agent.ipynb Cell 10 into orchestrator.
# ══════════════════════════════════════════════════════════════════════════════

import time as _time


def _safe_api(url, params=None, retries=3, delay=2):
    for i in range(retries):
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 200:
                return r.json()
        except Exception:
            pass
        _time.sleep(delay)
    return None


def get_structural_druggability_score(uniprot_id: str) -> dict:
    """
    RCSB PDB structural druggability score.
    Score: 4=X-ray+drug ligand, 3=X-ray+any ligand, 2=X-ray only, 1=AlphaFold, 0=none
    """
    BAD_LIGANDS = {"HOH","NA","CL","K","MG","CA","ZN","SO4","PO4",
                   "GOL","EDO","PEG","DMS","MPD","FMT","ACT","IOD"}
    search_url = "https://search.rcsb.org/rcsbsearch/v2/query"
    query = {
        "query": {"type": "terminal", "service": "text",
                  "parameters": {
                      "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                      "operator": "exact_match", "value": uniprot_id}},
        "return_type": "entry", "request_options": {"return_all_hits": True}
    }
    try:
        resp = requests.post(search_url, json=query, timeout=15)
        if resp.status_code != 200:
            return {"druggability_score":0,"pdb_count":0,"best_resolution":None,"has_drug_ligand":False}
        pdb_ids = [r["identifier"] for r in resp.json().get("result_set", [])]
    except Exception:
        return {"druggability_score":0,"pdb_count":0,"best_resolution":None,"has_drug_ligand":False}

    if not pdb_ids:
        try:
            af = requests.head(f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb", timeout=10)
            if af.status_code == 200:
                return {"druggability_score":1,"pdb_count":0,"best_resolution":None,"has_drug_ligand":False}
        except Exception:
            pass
        return {"druggability_score":0,"pdb_count":0,"best_resolution":None,"has_drug_ligand":False}

    best_res = 99.0; has_drug_lig = False; has_any_lig = False; has_xray = False
    for pdb_id in pdb_ids[:15]:
        entry = _safe_api(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
        if not entry: continue
        try:
            if "X-RAY" in entry["rcsb_entry_info"]["experimental_method"].upper():
                has_xray = True
        except Exception: pass
        try:
            res = entry["rcsb_entry_info"]["resolution_combined"][0]
            if res < best_res: best_res = res
        except Exception: pass
        try:
            ligs = entry["rcsb_entry_container_identifiers"]["non_polymer_entity_ids"]
            if [l for l in ligs if l not in BAD_LIGANDS]:
                has_any_lig = True; has_drug_lig = True
        except Exception: pass

    if has_xray and has_drug_lig:  score = 4
    elif has_xray and has_any_lig: score = 3
    elif has_xray:                 score = 2
    else:                          score = 1

    return {"druggability_score": score, "pdb_count": len(pdb_ids),
            "best_resolution": round(best_res, 2) if best_res < 99 else None,
            "has_drug_ligand": has_drug_lig}


def get_pubmed_literature_score(gene_name: str, disease_name: str) -> dict:
    """
    PubMed co-publication count for gene + disease.
    Score: 4=>100, 3=21-100, 2=6-20, 1=1-5, 0=none
    """
    if not gene_name or not disease_name:
        return {"pub_count": 0, "literature_score": 0}
    params = {
        "db": "pubmed",
        "term": f'("{gene_name}"[Title/Abstract]) AND ("{disease_name}"[Title/Abstract])',
        "retmode": "json", "retmax": 0,
        "tool": "biomedical_agent", "email": "research@example.com"
    }
    data = _safe_api("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", params=params)
    count = 0
    try: count = int(data["esearchresult"]["count"])
    except Exception: pass
    if count == 0:     score = 0
    elif count <= 5:   score = 1
    elif count <= 20:  score = 2
    elif count <= 100: score = 3
    else:              score = 4
    return {"pub_count": count, "literature_score": score}


def prioritize_targets_all_methods(disease: dict, targets: list = None, top_k: int = 10) -> dict:
    """
    Exact copy from obj_1_kg_agent.ipynb.
    Method 3: Structural Druggability (RCSB PDB)
    Method 4: Literature Evidence (PubMed)
    Consensus: top-5 in BOTH methods.
    """
    disease_name = disease.get("disease_name", "")
    if targets is None:
        targets = kg_disease_to_targets_direct(disease)["targets"]

    scored = []
    for t in targets:
        uniprot_id   = t["protein_id"]
        gene_name    = t.get("gene_name", "") or t.get("protein_name", "")
        protein_name = t.get("protein_name", uniprot_id)

        print(f"     Scoring {gene_name} ({uniprot_id})...")
        struct = get_structural_druggability_score(uniprot_id)
        lit    = get_pubmed_literature_score(gene_name, disease_name)
        _time.sleep(0.3)

        scored.append({
            "rank_pdb":         0,
            "rank_pubmed":      0,
            "protein_id":       uniprot_id,
            "gene_name":        gene_name,
            "protein_name":     protein_name,
            "structural_score": struct["druggability_score"],
            "pdb_count":        struct["pdb_count"],
            "best_resolution":  struct["best_resolution"],
            "has_drug_ligand":  struct["has_drug_ligand"],
            "pubmed_count":     lit["pub_count"],
            "literature_score": lit["literature_score"],
        })

    pdb_ranked = sorted(scored,
                        key=lambda x: (x["structural_score"],
                                       x["pdb_count"],
                                       -(x["best_resolution"] or 99)),
                        reverse=True)
    for rank, t in enumerate(pdb_ranked, 1):
        t["rank_pdb"] = rank

    pubmed_ranked = sorted(scored, key=lambda x: x["pubmed_count"], reverse=True)
    for rank, t in enumerate(pubmed_ranked, 1):
        t["rank_pubmed"] = rank

    top_n          = min(5, top_k)
    top_pdb_ids    = {t["protein_id"] for t in pdb_ranked[:top_n]}
    top_pubmed_ids = {t["protein_id"] for t in pubmed_ranked[:top_n]}
    consensus_ids  = top_pdb_ids & top_pubmed_ids
    consensus      = sorted(
        [t for t in scored if t["protein_id"] in consensus_ids],
        key=lambda x: x["rank_pdb"] + x["rank_pubmed"]
    )

    return {
        "method_3_structural_druggability": pdb_ranked[:top_k],
        "method_4_literature_evidence":     pubmed_ranked[:top_k],
        "consensus_targets":                consensus,
    }


# ─── Enhanced Pathway Functions (missing from orchestrator) ──────────────────

def kg_get_pathway_proteins(pathway_id: str) -> list:
    proteins = []
    for nbr in knowledge_graph.neighbors(pathway_id):
        node = knowledge_graph.nodes[nbr]
        if node.get("entity_type") == "protein":
            proteins.append({"protein_id": nbr,
                             "protein_name": node.get("protein_name", nbr),
                             "gene_name": node.get("gene_name", "")})
    return proteins


def kg_get_disease_pathway_set(disease_id: str) -> set:
    pathways = set()
    for protein in knowledge_graph.neighbors(disease_id):
        if knowledge_graph.nodes[protein].get("entity_type") != "protein":
            continue
        for node in knowledge_graph.neighbors(protein):
            if knowledge_graph.nodes[node].get("entity_type") == "pathway":
                pathways.add(node)
    return pathways


def kg_cross_disease_pathways(diseases: list, limit=None) -> dict:
    disease_pathway_sets = {}
    for disease in diseases:
        did  = disease["disease_id"]
        name = disease["disease_name"]
        disease_pathway_sets[did] = {"name": name, "pathways": kg_get_disease_pathway_set(did)}

    pathway_counts = {}
    for did, info in disease_pathway_sets.items():
        for pw_id in info["pathways"]:
            if pw_id not in pathway_counts:
                pathway_counts[pw_id] = {"diseases": [], "count": 0}
            pathway_counts[pw_id]["diseases"].append(info["name"])
            pathway_counts[pw_id]["count"] += 1

    shared = {pw_id: info for pw_id, info in pathway_counts.items() if info["count"] >= 2}

    results = []
    for pw_id, info in shared.items():
        node_data = knowledge_graph.nodes.get(pw_id, {})
        results.append({
            "pathway_id":      pw_id,
            "pathway_name":    node_data.get("pathway_name", pw_id),
            "source":          node_data.get("source", ""),
            "shared_count":    info["count"],
            "shared_diseases": info["diseases"]
        })

    results = sorted(results, key=lambda x: x["shared_count"], reverse=True)
    if limit not in [None, "ALL"]:
        results = results[:int(limit)]
    return {"diseases": [d["disease_name"] for d in diseases],
            "shared_pathways": results, "total_shared": len(results)}


def kg_pathway_to_diseases(pathway: dict, limit=None) -> dict:
    pathway_id = pathway["pathway_id"]
    diseases   = {}
    for protein in knowledge_graph.neighbors(pathway_id):
        if knowledge_graph.nodes[protein].get("entity_type") != "protein":
            continue
        for disease in knowledge_graph.neighbors(protein):
            node = knowledge_graph.nodes[disease]
            if node.get("entity_type") == "disease":
                diseases[disease] = {"disease_id": disease,
                                     "disease_name": node.get("disease_name", disease)}
    results = list(diseases.values())
    if limit not in [None, "ALL"]:
        results = results[:int(limit)]
    return {"pathway": pathway, "diseases": results, "count": len(results)}


def kg_pathway_hub_analysis(top_k: int = 20) -> dict:
    pathway_nodes = [(node, data) for node, data in knowledge_graph.nodes(data=True)
                     if data.get("entity_type") == "pathway"]
    hub_scores = []
    for pw_id, pw_data in pathway_nodes:
        diseases_connected = set()
        proteins_connected = set()
        for nbr in knowledge_graph.neighbors(pw_id):
            if knowledge_graph.nodes[nbr].get("entity_type") == "protein":
                proteins_connected.add(nbr)
                for nbr2 in knowledge_graph.neighbors(nbr):
                    if knowledge_graph.nodes[nbr2].get("entity_type") == "disease":
                        diseases_connected.add(nbr2)
        if diseases_connected:
            hub_scores.append({
                "pathway_id":    pw_id,
                "pathway_name":  pw_data.get("pathway_name", pw_id),
                "source":        pw_data.get("source", ""),
                "disease_count": len(diseases_connected),
                "protein_count": len(proteins_connected),
                "hub_score":     round(len(diseases_connected) / max(len(proteins_connected), 1), 3)
            })
    hub_scores = sorted(hub_scores, key=lambda x: x["disease_count"], reverse=True)
    return {"pathway_hubs": hub_scores[:top_k], "total_pathways_checked": len(pathway_nodes)}


def kg_pathway_disease_burden(pathway: dict) -> dict:
    pathway_id = pathway["pathway_id"]
    proteins   = kg_get_pathway_proteins(pathway_id)
    all_diseases = {}
    for p in proteins:
        for nbr in knowledge_graph.neighbors(p["protein_id"]):
            node = knowledge_graph.nodes[nbr]
            if node.get("entity_type") == "disease":
                all_diseases[nbr] = node.get("disease_name", nbr)
    return {"pathway": pathway, "protein_count": len(proteins),
            "disease_count": len(all_diseases),
            "diseases": [{"disease_id": did, "disease_name": dname}
                         for did, dname in all_diseases.items()]}


def kg_pathway_literature_validation(pathway_name: str, disease_name: str = "") -> dict:
    import time

    def pubmed_search(query_term):
        base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        params = {"db": "pubmed", "term": query_term, "retmode": "json", "retmax": 0,
                  "tool": "biomedical_agent", "email": "research@example.com"}
        try:
            resp = requests.get(base_url, params=params, timeout=15)
            if resp.status_code == 200:
                return int(resp.json()["esearchresult"]["count"])
        except Exception:
            pass
        return 0

    pw_count = pubmed_search(f'"{pathway_name}"[Title/Abstract]')
    time.sleep(0.5)
    combined_count = 0
    if disease_name:
        combined_count = pubmed_search(
            f'("{pathway_name}"[Title/Abstract]) AND ("{disease_name}"[Title/Abstract])')
        time.sleep(0.5)

    if combined_count > 100:   lit_score = 4
    elif combined_count > 20:  lit_score = 3
    elif combined_count > 5:   lit_score = 2
    elif combined_count > 0:   lit_score = 1
    else:                      lit_score = 0

    return {"pathway_name": pathway_name, "disease_name": disease_name,
            "pathway_pub_count": pw_count, "combined_pub_count": combined_count,
            "literature_score": lit_score,
            "validation": "Strong" if lit_score >= 3 else
                          "Moderate" if lit_score == 2 else
                          "Weak" if lit_score == 1 else "Not validated"}


def kg_disease_pathways_with_literature(resolved_disease: dict, limit=None) -> dict:
    import time
    basic    = kg_disease_to_pathways(resolved_disease, limit=None)
    pathways = basic["pathways"]
    print(f"   Validating {len(pathways)} pathways with PubMed...")
    disease_name = resolved_disease["disease_name"]
    validated = []
    for i, pw in enumerate(pathways):
        lit = kg_pathway_literature_validation(pw["pathway_name"], disease_name)
        time.sleep(0.3)
        validated.append({**pw,
                          "pathway_pub_count":  lit["pathway_pub_count"],
                          "combined_pub_count": lit["combined_pub_count"],
                          "literature_score":   lit["literature_score"],
                          "validation":         lit["validation"]})
        if (i + 1) % 5 == 0:
            print(f"   Validated {i+1}/{len(pathways)} pathways...")
    validated = sorted(validated, key=lambda x: x["literature_score"], reverse=True)
    if limit not in [None, "ALL"]:
        validated = validated[:int(limit)]
    return {"disease": resolved_disease, "pathways": validated,
            "total": len(validated),
            "strongly_validated": [p for p in validated if p["literature_score"] >= 3],
            "not_validated":      [p for p in validated if p["literature_score"] == 0]}


# ─── FULL format_results_plain_text from obj-1 (replaces stripped version) ───

def format_results_plain_text(results, intent):
    if intent == "DISEASE_TO_TARGET":
        disease = results["disease"]["disease_name"]
        text = f"\n🧬 Targets associated with {disease}:\n\n"
        for i, t in enumerate(results.get("all_targets", []), 1):
            text += f"{i}. {t['gene_name']} ({t['protein_id']})\n"

        ranked = results.get("ranked_targets", {})
        if ranked and "method_3_structural_druggability" in ranked:
            text += "\n\n📊 Table 1 — Structural Druggability Ranking (RCSB PDB)\n"
            text += "   Score: 4=X-ray+drug ligand  3=X-ray+any ligand  2=X-ray only  1=AlphaFold  0=none\n"
            text += f"   {'Rank':<6}{'UniProt':<12}{'Gene':<12}{'Score':<8}{'PDBs':<8}{'Best Res(Å)':<14}{'Drug Ligand'}\n"
            text += "   " + "-"*68 + "\n"
            for t in ranked.get("method_3_structural_druggability", []):
                res = f"{t['best_resolution']}Å" if t['best_resolution'] else "N/A"
                lig = "Yes" if t['has_drug_ligand'] else "No"
                text += (f"   {t['rank_pdb']:<6}{t['protein_id']:<12}"
                         f"{t['gene_name']:<12}{t['structural_score']:<8}"
                         f"{t['pdb_count']:<8}{res:<14}{lig}\n")

            text += "\n\n📊 Table 2 — Literature Evidence Ranking (PubMed)\n"
            text += "   Score: 4=>100 papers  3=21-100  2=6-20  1=1-5  0=none\n"
            text += f"   {'Rank':<6}{'UniProt':<12}{'Gene':<12}{'Score':<8}{'Publications'}\n"
            text += "   " + "-"*52 + "\n"
            for t in ranked.get("method_4_literature_evidence", []):
                text += (f"   {t['rank_pubmed']:<6}{t['protein_id']:<12}"
                         f"{t['gene_name']:<12}{t['literature_score']:<8}"
                         f"{t['pubmed_count']}\n")

            consensus = ranked.get("consensus_targets", [])
            text += "\n\n★  Consensus Targets (top-5 in both methods):\n"
            if consensus:
                for j, t in enumerate(consensus, 1):
                    text += (f"   {j}. {t['gene_name']} ({t['protein_id']}) "
                             f"— PDB rank #{t['rank_pdb']}, "
                             f"PubMed rank #{t['rank_pubmed']} "
                             f"({t['pubmed_count']} papers)\n")
            else:
                text += "   No protein in top-5 of both methods\n"
        return text

    elif intent == "DISEASE_TO_PATHWAY":
        disease = results["disease"]["disease_name"]
        text = f"\n🧪 Pathways involved in {disease}:\n\n"
        for i, p in enumerate(results.get("pathways", []), 1):
            text += f"{i}. {p['pathway_name']}\n"
        return text

    elif intent == "TARGET_TO_PATHWAY":
        protein = results.get("protein", {}).get("gene_name", "")
        text = f"\n🧬 Pathways involving {protein}:\n\n"
        for i, p in enumerate(results.get("pathways", []), 1):
            text += f"{i}. {p['pathway_name']}\n"
        return text

    elif intent == "DISEASE_PPI_NETWORK":
        disease = results["disease"]["disease_name"]
        summary = results.get("ppi_network_summary", {})
        return (f"\n🔗 PPI Network for {disease}:\n\n"
                f"Proteins: {summary.get('num_proteins', 0)}\n"
                f"Interactions: {summary.get('num_interactions', 0)}\n")

    elif intent == "DISEASE_PPI_HUBS":
        disease = results["disease"]["disease_name"]
        text = f"\n⭐ Hub proteins in {disease} PPI network:\n\n"
        for i, p in enumerate(results.get("hub_proteins", []), 1):
            pid  = p.get("protein_id", "")
            gene = knowledge_graph.nodes[pid].get("gene_name", pid) if pid in knowledge_graph.nodes else pid
            text += f"{i}. {gene} (centrality: {round(p.get('score', 0), 4)})\n"
        return text

    elif intent == "PATHWAY_LITERATURE_VALIDATION":
        disease  = results.get("disease", {}).get("disease_name", "")
        pathways = results.get("pathways", [])
        strong   = results.get("strongly_validated", [])
        not_val  = results.get("not_validated", [])
        text  = f"\n📚 Literature-Validated Pathways for {disease}:\n"
        text += f"   Total: {len(pathways)} | Strongly validated: {len(strong)} | Not validated: {len(not_val)}\n\n"
        text += f"   {'Rank':<5}{'Pathway':<50}{'Source':<12}{'Pubs':<8}{'Score':<7}{'Validation'}\n"
        text += "   " + "-"*90 + "\n"
        for i, p in enumerate(pathways[:20], 1):
            name = p["pathway_name"][:48] + ("..." if len(p["pathway_name"]) > 48 else "")
            text += (f"   {i:<5}{name:<50}{p.get('source', ''):<12}"
                     f"{p.get('combined_pub_count', 0):<8}"
                     f"{p.get('literature_score', 0):<7}{p.get('validation', '')}\n")
        return text

    elif intent == "CROSS_DISEASE_PATHWAYS":
        diseases = results.get("diseases", [])
        shared   = results.get("shared_pathways", [])
        total    = results.get("total_shared", 0)
        text  = f"\n🔀 Cross-Disease Shared Pathways\n"
        text += f"   Diseases: {', '.join(diseases)}\n"
        text += f"   Total shared pathways: {total}\n\n"
        text += f"   {'Rank':<5}{'Pathway':<50}{'Source':<12}{'Shared by'}\n"
        text += "   " + "-"*80 + "\n"
        for i, p in enumerate(shared[:20], 1):
            name = p["pathway_name"][:48] + ("..." if len(p["pathway_name"]) > 48 else "")
            text += (f"   {i:<5}{name:<50}{p.get('source', ''):<12}"
                     f"{p['shared_count']} diseases\n")
        return text

    elif intent == "PATHWAY_HUB_ANALYSIS":
        hubs  = results.get("pathway_hubs", [])
        total = results.get("total_pathways_checked", 0)
        text  = f"\n⭐ Pathway Hub Analysis ({total} pathways checked)\n\n"
        text += f"   {'Rank':<5}{'Pathway':<50}{'Source':<12}{'Diseases':<10}{'Proteins':<10}{'Hub Score'}\n"
        text += "   " + "-"*95 + "\n"
        for i, p in enumerate(hubs[:20], 1):
            name = p["pathway_name"][:48] + ("..." if len(p["pathway_name"]) > 48 else "")
            text += (f"   {i:<5}{name:<50}{p.get('source', ''):<12}"
                     f"{p['disease_count']:<10}{p['protein_count']:<10}"
                     f"{p['hub_score']}\n")
        return text

    elif intent in ("PATHWAY_TO_DISEASES", "PATHWAY_DISEASE_BURDEN"):
        pathway  = results.get("pathway", {})
        diseases = results.get("diseases", [])
        count    = results.get("count", results.get("disease_count", 0))
        text  = f"\n🧬 Diseases associated with: {pathway.get('pathway_name', '')}\n"
        text += f"   Total: {count}\n\n"
        for i, d in enumerate(diseases[:20], 1):
            text += f"   {i}. {d['disease_name']}\n"
        return text

    elif intent == "DRUG_TO_TARGET":
        drug    = results.get("drug", {})
        targets = results.get("targets", [])
        text  = f"\n💊 Targets of {drug.get('drug_name', drug.get('drug_id', ''))}:\n\n"
        for i, t in enumerate(targets, 1):
            text += f"{i}. {t.get('gene_name', '')} ({t.get('protein_id', '')})\n"
        return text

    elif intent in ("MOLECULE_TO_TARGET", "MOLECULE_TO_DISEASE", "MOLECULE_TO_PATHWAY"):
        mol     = results.get("molecule", {})
        targets = results.get("targets", [])
        text  = f"\n🔬 Targets of {mol.get('molecule_name', mol.get('molecule_id', ''))}:\n\n"
        for i, t in enumerate(targets, 1):
            text += f"{i}. {t.get('gene_name', '')} ({t.get('protein_id', '')})\n"
        return text

    else:
        return str(results)


# ─── FULL explain_results_with_llm from obj-1 ────────────────────────────────

def explain_results_with_llm(query, results, intent):
    system = "You are a biomedical research assistant explaining computational biology results."
    common_instruction = """
Instructions:
- Explain ONLY using the provided results (no generic biology)
- For each important entity, explain its role using given data
- If ranking is provided, explain WHY top items are ranked higher
- Identify biological patterns

Output:
- 6-8 lines explanation
- Mention specific targets/pathways
- Include reasoning (NOT just description)
"""
    if intent == "DISEASE_TO_TARGET":
        targets   = [t["gene_name"] for t in results.get("all_targets", [])]
        ranked    = results.get("ranked_targets", {})
        consensus = ranked.get("consensus_targets", []) if ranked else []
        m3 = [(t["gene_name"], t["protein_id"], t["structural_score"],
               t["pdb_count"], t["best_resolution"])
              for t in ranked.get("method_3_structural_druggability", [])[:5]] if ranked else []
        m4 = [(t["gene_name"], t["protein_id"], t["pubmed_count"])
              for t in ranked.get("method_4_literature_evidence", [])[:5]] if ranked else []
        user = (f"Query: {query}\n\nDisease: {results['disease']['disease_name']}\n"
                f"All KG Targets: {targets}\n\n"
                f"Method 3 - Structural Druggability (RCSB PDB):\n"
                f"(gene, uniprot, score 0-4, num_PDBs, best_resolution_angstrom)\n{m3}\n\n"
                f"Method 4 - Literature Evidence (PubMed):\n"
                f"(gene, uniprot, publication_count)\n{m4}\n\n"
                f"Consensus Targets (top-5 of both): {[t['gene_name'] for t in consensus]}\n\n"
                f"{common_instruction}")

    elif intent == "DISEASE_TO_PATHWAY":
        pathways = [p["pathway_name"] for p in results.get("pathways", [])]
        user = (f"Query: {query}\n\nDisease: {results['disease']['disease_name']}\n"
                f"Pathways:\n{pathways}\n\n{common_instruction}")

    elif intent == "TARGET_TO_PATHWAY":
        protein  = results.get("protein", {}).get("gene_name", "")
        pathways = [p["pathway_name"] for p in results.get("pathways", [])]
        user = (f"Query: {query}\n\nProtein: {protein}\n"
                f"Pathways:\n{pathways}\n\n{common_instruction}")

    elif intent == "DISEASE_PPI_NETWORK":
        user = (f"Query: {query}\n\nNetwork summary:\n{results.get('ppi_network_summary', {})}\n"
                f"Proteins:\n{results.get('ppi_network_proteins', [])}\n\n{common_instruction}")

    elif intent == "DISEASE_PPI_HUBS":
        hubs = results.get("hub_proteins", [])
        enriched = []
        for h in hubs:
            pid  = h.get("protein_id", "")
            gene = knowledge_graph.nodes[pid].get("gene_name", pid) if pid in knowledge_graph.nodes else pid
            enriched.append({"gene": gene, "score": h.get("score", 0)})
        user = (f"Query: {query}\n\nHub proteins (centrality scores):\n{enriched}\n\n{common_instruction}")

    elif intent == "PATHWAY_LITERATURE_VALIDATION":
        pathways = [(p["pathway_name"], p.get("combined_pub_count", 0), p.get("validation", ""))
                    for p in results.get("pathways", [])[:10]]
        user = (f"Query: {query}\n\nDisease: {results.get('disease', {}).get('disease_name', '')}\n"
                f"Validated Pathways (name, pub_count, validation_strength):\n{pathways}\n\n{common_instruction}")

    elif intent == "CROSS_DISEASE_PATHWAYS":
        shared = [(p["pathway_name"], p["shared_count"], p["shared_diseases"])
                  for p in results.get("shared_pathways", [])[:10]]
        user = (f"Query: {query}\n\nDiseases compared: {results.get('diseases', [])}\n"
                f"Shared pathways (name, shared_by_N_diseases, which_diseases):\n{shared}\n\n{common_instruction}")

    elif intent == "PATHWAY_HUB_ANALYSIS":
        hubs = [(p["pathway_name"], p["disease_count"], p["protein_count"], p["hub_score"])
                for p in results.get("pathway_hubs", [])[:10]]
        user = (f"Query: {query}\n\nTop pathway hubs (name, disease_count, protein_count, hub_score):\n"
                f"{hubs}\n\n{common_instruction}")

    elif intent in ("PATHWAY_TO_DISEASES", "PATHWAY_DISEASE_BURDEN"):
        user = (f"Query: {query}\n\nPathway: {results.get('pathway', {}).get('pathway_name', '')}\n"
                f"Diseases connected: {[d['disease_name'] for d in results.get('diseases', [])[:15]]}\n\n"
                f"{common_instruction}")

    else:
        user = f"Query: {query}\n\nResults:\n{results}\n\n{common_instruction}"

    return call_ollama("llama3:8b", system, user)


print("✅ All missing obj-1 functions loaded:")
print("   ✔ get_structural_druggability_score()")
print("   ✔ get_pubmed_literature_score()")
print("   ✔ prioritize_targets_all_methods()  ← FIXED")
print("   ✔ kg_cross_disease_pathways()")
print("   ✔ kg_pathway_to_diseases()")
print("   ✔ kg_pathway_hub_analysis()")
print("   ✔ kg_pathway_disease_burden()")
print("   ✔ kg_pathway_literature_validation()")
print("   ✔ kg_disease_pathways_with_literature()")
print("   ✔ format_results_plain_text()       ← FULL version")
print("   ✔ explain_results_with_llm()        ← FULL version")


✅ All missing obj-1 functions loaded:
   ✔ get_structural_druggability_score()
   ✔ get_pubmed_literature_score()
   ✔ prioritize_targets_all_methods()  ← FIXED
   ✔ kg_cross_disease_pathways()
   ✔ kg_pathway_to_diseases()
   ✔ kg_pathway_hub_analysis()
   ✔ kg_pathway_disease_burden()
   ✔ kg_pathway_literature_validation()
   ✔ kg_disease_pathways_with_literature()
   ✔ format_results_plain_text()       ← FULL version
   ✔ explain_results_with_llm()        ← FULL version


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# ORCHESTRATOR STATE + NODES  (speed-fixed + correctness-fixed)
#
# SPEED: node_route_query does ALL preprocessing once (normalize, spell-correct,
#        detect_intent, extract_entities, optimize_query). node_kg_agent reads
#        from state — no repeated LLM calls.
#        Was: 10-11 LLM calls/query.  Now: 5-6 LLM calls/query.
#
# CORRECTNESS: node_kg_agent now calls the real functions from obj-1 directly
#              (loaded in cell above) instead of routing through kg_agent.invoke()
#              which was re-running preprocessing and using broken stub functions.
# ══════════════════════════════════════════════════════════════════════════════

class OrchestratorState(TypedDict):
    query:            str
    corrected_query:  str
    intent:           str
    route:            str
    entities:         dict
    normalized:       dict
    limit:            Any
    kg_output:        dict
    structure_output: dict
    final_answer:     str
    messages:         Annotated[list, operator.add]


# ── NODE 1: Route Query ───────────────────────────────────────────────────────

def node_route_query(state: OrchestratorState) -> OrchestratorState:
    query = state["query"]

    # Step 0: normalize informal query (1 LLM call)
    query = normalize_natural_query(query)

    # Step 1: spell-correct entity names (1 LLM call)
    corrected = correct_query_spelling_and_entities(query)

    # Step 2: detect intent (1 LLM call)
    intent = detect_intent(corrected).strip()

    # Step 3: classify route
    route = classify_route(intent)

    # Step 4: extract + resolve entities (1 LLM call) — done ONCE here
    entities   = extract_entities(corrected)
    normalized = normalize_entities_kg_grounded(entities)
    normalized["Protein"] = [p for p in normalized.get("Protein", [])
                              if p.get("confidence") == "high" or p.get("match_score", 0) >= 80]

    # Step 5: result-size limit (1 LLM call)
    constraints = optimize_query(corrected)
    limit       = constraints.get("limit", 10)
    m = re.search(r'\btop\s*(\d+)', corrected.lower())
    if m: limit = int(m.group(1))

    print(f"\n🔀 Router | Intent: {intent} | Route: {route}")
    print(f"   Corrected: {corrected}")
    print(f"   Entities:  {entities}")

    return {
        **state,
        "query":           corrected,
        "corrected_query": corrected,
        "intent":          intent,
        "route":           route,
        "entities":        entities,
        "normalized":      normalized,
        "limit":           limit,
        "messages":        [AIMessage(content=f"Intent: {intent} | Route: {route}")]
    }


def route_after_router(state: OrchestratorState) -> str:
    return state["route"]


# ── NODE 2: KG Agent ─────────────────────────────────────────────────────────
# Calls obj-1 functions directly — no kg_agent.invoke(), no repeat preprocessing.

def node_kg_agent(state: OrchestratorState) -> OrchestratorState:
    print("\n" + "═"*50)
    print("  RUNNING KG AGENT")
    print("═"*50)

    corrected  = state["corrected_query"]
    intent     = state["intent"]
    normalized = state["normalized"]
    entities   = state["entities"]
    limit      = state["limit"]
    results    = {}

    print(f"\n🧬 KG Query | Intent: {intent}")

    if intent == "DISEASE_TO_TARGET" and normalized.get("Disease"):
        disease     = normalized["Disease"][0]
        raw_targets = kg_disease_to_targets_direct(disease, limit="ALL")["targets"]
        print(f"\n🧬 Targets associated with {disease['disease_name']}:")
        for i, t in enumerate(raw_targets, 1):
            print(f"  {i}. {t['gene_name']}")
        print(f"\n🔬 Target Prioritization (scoring {len(raw_targets)} targets):")
        ranked  = prioritize_targets_all_methods(disease, targets=raw_targets, top_k=10)
        results = {"disease": disease, "all_targets": raw_targets, "ranked_targets": ranked}

    elif intent == "DISEASE_TO_PATHWAY" and normalized.get("Disease"):
        results = kg_disease_to_pathways(normalized["Disease"][0], limit)

    elif intent == "TARGET_TO_PATHWAY" and normalized.get("Protein"):
        protein = normalized["Protein"][0]
        results = {"protein": protein,
                   "pathways": kg_target_to_pathways(protein["protein_id"], limit)}

    elif intent == "DRUG_TO_TARGET" and normalized.get("Drug"):
        drug    = normalized["Drug"][0]
        targets = kg_drug_to_targets(drug["drug_id"], limit)
        results = {"drug": drug, "targets": targets}

    elif intent == "MOLECULE_TO_TARGET" and normalized.get("Molecule"):
        mol     = normalized["Molecule"][0]
        targets = kg_molecule_to_targets(mol["molecule_id"], limit)
        results = {"molecule": mol, "targets": targets}

    elif intent == "DISEASE_PPI_NETWORK" and normalized.get("Disease"):
        disease    = normalized["Disease"][0]
        disease_id = disease["disease_id"]
        G_ppi      = build_disease_ppi_subgraph(disease_id)
        if G_ppi.number_of_nodes() > 0:
            net  = plot_ppi_network_interactive(G_ppi, disease_id=disease_id, notebook=True)
            html = net.generate_html()
            display(HTML(html))
            with open(f"ppi_{disease_id}.html", "w", encoding="utf-8") as f:
                f.write(html)
            results = {"disease": disease,
                       "ppi_network_summary": {"num_proteins":     G_ppi.number_of_nodes(),
                                               "num_interactions": G_ppi.number_of_edges()},
                       "ppi_network_proteins": list(G_ppi.nodes())}
        else:
            results = {"disease": disease, "message": "No PPI interactions found"}

    elif intent == "DISEASE_PPI_HUBS" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        results = {"disease": disease,
                   "hub_proteins": kg_disease_ppi_hubs(
                       disease, int(limit) if isinstance(limit, int) else 10)}

    elif intent in ("DISEASE_DRUGGABLE_TARGETS", "FULL_PIPELINE") and normalized.get("Disease"):
        disease     = normalized["Disease"][0]
        raw_targets = kg_disease_to_targets_direct(disease, limit="ALL")["targets"]
        ranked      = prioritize_targets_all_methods(disease, targets=raw_targets, top_k=10)
        results     = {"disease": disease, "all_targets": raw_targets, "ranked_targets": ranked}

    elif intent == "PATHWAY_LITERATURE_VALIDATION" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        print(f"   Validating pathways for {disease['disease_name']}...")
        results = kg_disease_pathways_with_literature(disease, limit=limit)

    elif intent == "CROSS_DISEASE_PATHWAYS" and normalized.get("Disease"):
        disease1 = normalized["Disease"][0]
        diseases = [disease1]
        for dname in entities.get("Disease", [])[1:]:
            resolved2 = resolve_disease_kg_grounded(dname)
            if resolved2:
                diseases.append(resolved2)
        results = kg_cross_disease_pathways(diseases, limit=limit)

    elif intent == "PATHWAY_HUB_ANALYSIS":
        top_k = int(limit) if isinstance(limit, int) else 20
        results = kg_pathway_hub_analysis(top_k=top_k)

    elif intent == "PATHWAY_TO_DISEASES" and normalized.get("Pathway"):
        results = kg_pathway_to_diseases(normalized["Pathway"][0], limit=limit)

    elif intent == "PATHWAY_DISEASE_BURDEN" and normalized.get("Pathway"):
        results = kg_pathway_disease_burden(normalized["Pathway"][0])

    else:
        results = {"message": "No valid result found. Check entity extraction or intent."}

    formatted   = format_results_plain_text(results, intent)
    print(formatted)

    # 1 LLM call for explanation
    explanation = explain_results_with_llm(corrected, results, intent)
    print("\n--- LLM Explanation ---")
    print(explanation)

    # Build druggable targets list for full pipeline
    druggable_targets = []
    if intent in ("DISEASE_TO_TARGET", "DISEASE_DRUGGABLE_TARGETS", "FULL_PIPELINE"):
        ranked    = results.get("ranked_targets", {})
        consensus = ranked.get("consensus_targets", [])
        top_m3    = ranked.get("method_3_structural_druggability", [])[:3]
        top_pr    = consensus if consensus else top_m3
        druggable_targets = [{"protein_id":  t["protein_id"],
                              "gene_name":    t.get("gene_name", ""),
                              "protein_name": t.get("protein_name", "")}
                             for t in top_pr[:3]]

    shared_payload = {
        "query":             corrected,
        "intent":            intent,
        "disease":           results.get("disease", {}),
        "druggable_targets": druggable_targets,
        "kg_summary":        formatted,
        "llm_explanation":   explanation,
        "kg_results":        results,
    }
    with open(KG_OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(shared_payload, f, indent=2)
    print(f"\n✅ KG Agent complete | shared/kg_output.json written")
    print(f"   Druggable targets: {[t['gene_name'] for t in druggable_targets]}")

    return {
        **state,
        "kg_output": shared_payload,
        "messages":  [AIMessage(content=explanation)]
    }


# ── NODE 3: Structure Agent ───────────────────────────────────────────────────

def _run_structure_agent_subprocess(uniprot_id: str, protein_name: str = "") -> dict:
    task      = {"uniprot_id": uniprot_id, "protein_name": protein_name}
    task_file = os.path.join(SHARED_DIR, "obj2_task.json")
    with open(task_file, "w", encoding="utf-8") as f:
        json.dump(task, f, indent=2)
    print(f"\n📋 Task written: {task}")
    print("   Launching meeko_env subprocess...")
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"
    result = subprocess.run(
        [MEEKO_PYTHON, OBJ2_RUNNER_SCRIPT],
        capture_output=True, text=True,
        cwd=BASE_DIR, env=env, encoding="utf-8", errors="replace"
    )
    print("   STDOUT:\n", result.stdout[-2000:] if result.stdout else "(none)")
    if result.returncode != 0:
        print("   STDERR:\n", result.stderr[-1000:])
        return {"error": "Structure agent subprocess failed", "stderr": result.stderr[-500:]}
    if os.path.exists(STRUCTURE_OUTPUT_FILE):
        with open(STRUCTURE_OUTPUT_FILE, encoding="utf-8", errors="replace") as f:
            return json.load(f)
    return {"error": "structure_output.json not found"}


def node_structure_agent(state: OrchestratorState) -> OrchestratorState:
    print("\n" + "═"*50)
    print("  RUNNING STRUCTURE AGENT (meeko_env)")
    print("═"*50)

    query      = state["corrected_query"]
    normalized = state["normalized"]

    # Regex first — no LLM for explicit UniProt IDs
    UNIPROT_PAT = re.compile(r"\b[OPQ][0-9][A-Z0-9]{3}[0-9]\b")
    m = UNIPROT_PAT.search(query)
    if m:
        uniprot_id   = m.group()
        protein_name = uniprot_id
    elif normalized.get("Protein"):
        # Already resolved in node_route_query — no extra LLM call
        p            = normalized["Protein"][0]
        uniprot_id   = p["protein_id"]
        protein_name = p.get("gene_name", uniprot_id)
    else:
        msg = "❌ No protein/UniProt ID found. Include a UniProt ID (e.g. P07471)."
        print(msg)
        return {**state, "final_answer": msg, "messages": [AIMessage(content=msg)]}

    print(f"   UniProt: {uniprot_id} | Protein: {protein_name}")
    struct_output = _run_structure_agent_subprocess(uniprot_id, protein_name)
    print("\n✅ Structure Agent complete")
    return {
        **state,
        "structure_output": struct_output,
        "messages": [AIMessage(content=f"Structure analysis complete for {uniprot_id}")]
    }


# ── NODE 4: Full Pipeline ─────────────────────────────────────────────────────

def node_full_pipeline(state: OrchestratorState) -> OrchestratorState:
    print("\n" + "═"*50)
    print("  RUNNING FULL PIPELINE (KG → Structure)")
    print("═"*50)

    state             = node_kg_agent(state)
    kg_output         = state.get("kg_output", {})
    druggable_targets = kg_output.get("druggable_targets", [])
    print(f"\n   Druggable targets: {[t['gene_name'] for t in druggable_targets]}")

    all_struct_outputs = {}
    for target in druggable_targets[:3]:
        uid   = target["protein_id"]
        gname = target.get("gene_name", target.get("protein_name", ""))
        print(f"\n🔬 Structure: {uid} ({gname})")
        out = _run_structure_agent_subprocess(uid, gname)
        all_struct_outputs[uid] = out

    return {
        **state,
        "structure_output": all_struct_outputs,
        "messages": [AIMessage(content=f"Full pipeline: {len(all_struct_outputs)} proteins analysed")]
    }


# ── NODE 5: Synthesize ────────────────────────────────────────────────────────

def node_synthesize(state: OrchestratorState) -> OrchestratorState:
    route  = state["route"]
    intent = state["intent"]
    query  = state["corrected_query"]

    sections = [
        "# 🧬 Biomedical Agent Results",
        f"**Query:** {query}",
        f"**Intent:** `{intent}` | **Route:** `{route}`",
        "---"
    ]

    if route in ("kg", "full") and state.get("kg_output"):
        kg = state["kg_output"]
        sections += ["## 📊 Knowledge Graph Analysis",
                     kg.get("kg_summary", ""),
                     "### 🤖 LLM Explanation",
                     kg.get("llm_explanation", "")]
        if kg.get("druggable_targets"):
            sections.append("### 🎯 Top Druggable Targets")
            for t in kg["druggable_targets"]:
                sections.append(f"- **{t['gene_name']}** ({t['protein_id']})")
        sections.append("---")

    if route == "structure" and state.get("structure_output"):
        st = state["structure_output"]
        if "error" not in st:
            sections += ["## 🔬 Structural Analysis",
                         f"**Protein:** {st.get('uniprot_id','')} | **PDB:** {st.get('pdb_id','')}",
                         f"**Pockets:** {st.get('pockets_detected',0)} | **Ligands:** {st.get('ligands_retrieved',0)}"]
            for exp in st.get("pocket_explanations", []):
                sections.append(f"\n**{exp['pocket']}** ← `{exp['chembl_id']}` (score: {exp['docking_score']})")
                sections.append(exp.get("explanation_snippet", ""))
        else:
            sections.append(f"⚠️ {st.get('error', '')}")

    if route == "full" and isinstance(state.get("structure_output"), dict):
        for uid, st in state["structure_output"].items():
            if "error" in st: continue
            sections += [f"## 🔬 {uid}",
                         f"**PDB:** {st.get('pdb_id','')} | **Pockets:** {st.get('pockets_detected',0)} | **Ligands:** {st.get('ligands_retrieved',0)}"]
            for exp in st.get("pocket_explanations", [])[:2]:
                sections.append(f"\n**{exp['pocket']}** ← `{exp['chembl_id']}` (score: {exp['docking_score']})")
                sections.append(exp.get("explanation_snippet", ""))
            sections.append("---")

    final_answer = "\n\n".join(sections)
    return {**state, "final_answer": final_answer,
            "messages": [AIMessage(content="Results synthesized")]}


def node_display(state: OrchestratorState) -> OrchestratorState:
    display(Markdown(state["final_answer"]))
    return state


print("✅ All orchestrator nodes defined")


✅ All orchestrator nodes defined


In [18]:
orch_builder = StateGraph(OrchestratorState)

orch_builder.add_node("route_query", node_route_query)
orch_builder.add_node("kg",          node_kg_agent)
orch_builder.add_node("structure",   node_structure_agent)
orch_builder.add_node("full",        node_full_pipeline)
orch_builder.add_node("synthesize",  node_synthesize)
orch_builder.add_node("display",     node_display)

orch_builder.set_entry_point("route_query")

orch_builder.add_conditional_edges(
    "route_query",
    route_after_router,
    {"kg": "kg", "structure": "structure", "full": "full"}
)

orch_builder.add_edge("kg",         "synthesize")
orch_builder.add_edge("structure",  "synthesize")
orch_builder.add_edge("full",       "synthesize")
orch_builder.add_edge("synthesize", "display")
orch_builder.add_edge("display",    END)

orchestrator = orch_builder.compile()

print("✅ Orchestrator LangGraph compiled")
print("   Nodes:", list(orch_builder.nodes.keys()))


✅ Orchestrator LangGraph compiled
   Nodes: ['route_query', 'kg', 'structure', 'full', 'synthesize', 'display']


In [19]:
def run(query: str) -> dict:
    """
    Unified entry point for ALL query types.

    Examples:
      run("What are targets for Lung cancer?")
      run("Which all pathways involve in Lung cancer?")
      run("Which pathways involve the protein EGFR?")
      run("Which pathways does target P07471 participate in?")
      run("What are all targets for Type 2 diabetes mellitus disease?")
      run("Which proteins are hub nodes in the Alzheimer PPI network?")
      run("What proteins does Metformin act on?")
      run("Which pathways are shared between lung cancer and breast cancer?")
      run("Which pathways are the biggest hubs connecting the most diseases?")
      run("Find binding pockets for Q01469")
      run("Find drug candidates for EGFR")
      run("Perform docking for protein P07471")
      run("Identify druggable targets in Alzheimer's disease")
      run("Run full drug discovery pipeline for lung cancer")
    """
    print("\n" + "█"*60)
    print(f"  BIOMEDICAL AGENT  |  Query: {query}")
    print("█"*60)

    initial: OrchestratorState = {
        "query":            query,
        "corrected_query":  "",
        "intent":           "",
        "route":            "",
        "entities":         {},
        "normalized":       {},
        "limit":            10,
        "kg_output":        {},
        "structure_output": {},
        "final_answer":     "",
        "messages":         [HumanMessage(content=query)]
    }

    final = orchestrator.invoke(initial)

    print("\n" + "█"*60)
    print("  DONE")
    print("█"*60)
    return final


print("✅ run() ready")


✅ run() ready


## KG Queries

In [9]:
run("What are targets for Lung cancer?")


████████████████████████████████████████████████████████████
  BIOMEDICAL AGENT  |  Query: What are targets for Lung cancer?
████████████████████████████████████████████████████████████
   🔄 Natural query normalized:
      Input:  What are targets for Lung cancer?
      Output: What are targets for lung cancer?
Raw entity output: {"Disease":["lung cancer"],"Drug":[],"Protein":[],"Molecule":[],"Pathway":[]}
 Raw optimization output: { "limit": 10 }

🔀 Router | Intent: DISEASE_TO_TARGET | Route: kg
   Corrected: What are targets for lung cancer?
   Entities:  {'Disease': ['lung cancer'], 'Drug': [], 'Protein': [], 'Molecule': [], 'Pathway': []}

══════════════════════════════════════════════════
  RUNNING KG AGENT
══════════════════════════════════════════════════

🧬 KG Query | Intent: DISEASE_TO_TARGET

🧬 Targets associated with Lung cancer:
  1. DLEC1
  2. BRAF
  3. EGFR
  4. ERBB2
  5. SLC67A1
  6. MXRA5
  7. CASP8
  8. TP53

🔬 Target Prioritization (scoring 8 targets):
     Scoring 

# 🧬 Biomedical Agent Results

**Query:** What are targets for lung cancer?

**Intent:** `DISEASE_TO_TARGET` | **Route:** `kg`

---

## 📊 Knowledge Graph Analysis


🧬 Targets associated with Lung cancer:

1. DLEC1 (Q9Y238)
2. BRAF (P15056)
3. EGFR (P00533)
4. ERBB2 (P04626)
5. SLC67A1 (Q96BI1)
6. MXRA5 (Q9NR99)
7. CASP8 (Q14790)
8. TP53 (P04637)


📊 Table 1 — Structural Druggability Ranking (RCSB PDB)
   Score: 4=X-ray+drug ligand  3=X-ray+any ligand  2=X-ray only  1=AlphaFold  0=none
   Rank  UniProt     Gene        Score   PDBs    Best Res(Å)   Drug Ligand
   --------------------------------------------------------------------
   1     P00533      EGFR        4       385     2.4Å          Yes
   2     P04637      TP53        4       312     1.5Å          Yes
   3     P15056      BRAF        4       131     1.99Å         Yes
   4     P04626      ERBB2       4       63      1.25Å         Yes
   5     Q14790      CASP8       4       36      1.18Å         Yes
   6     Q9Y238      DLEC1       0       0       N/A           No
   7     Q96BI1      SLC67A1     0       0       N/A           No
   8     Q9NR99      MXRA5       0       0       N/A           No


📊 Table 2 — Literature Evidence Ranking (PubMed)
   Score: 4=>100 papers  3=21-100  2=6-20  1=1-5  0=none
   Rank  UniProt     Gene        Score   Publications
   ----------------------------------------------------
   1     P00533      EGFR        4       22123
   2     P04637      TP53        4       2310
   3     P15056      BRAF        4       1875
   4     P04626      ERBB2       4       857
   5     Q14790      CASP8       3       48
   6     Q9Y238      DLEC1       2       16
   7     Q9NR99      MXRA5       1       5
   8     Q96BI1      SLC67A1     0       0


★  Consensus Targets (top-5 in both methods):
   1. EGFR (P00533) — PDB rank #1, PubMed rank #1 (22123 papers)
   2. TP53 (P04637) — PDB rank #2, PubMed rank #2 (2310 papers)
   3. BRAF (P15056) — PDB rank #3, PubMed rank #3 (1875 papers)
   4. ERBB2 (P04626) — PDB rank #4, PubMed rank #4 (857 papers)
   5. CASP8 (Q14790) — PDB rank #5, PubMed rank #5 (48 papers)


### 🤖 LLM Explanation

Based on the computational biology results, lung cancer targets can be identified as follows:

The consensus targets, which are the top-5 genes from both structural druggability and literature evidence, are ['EGFR', 'TP53', 'BRAF', 'ERBB2', 'CASP8']. These genes play important roles in lung cancer.

* EGFR (Epidermal Growth Factor Receptor) is a key target with the highest score in both methods. It's ranked first due to its high structural druggability score of 4 and a large number of publications related to lung cancer (22,123). EGFR is a well-known oncogene that drives lung cancer development.
* TP53 (Tumor Protein P53) is another top target with a high structural druggability score of 4. It's ranked second due to its significant literature evidence (2,310 publications). TP53 is a tumor suppressor gene that plays a crucial role in preventing tumorigenesis.
* BRAF (B-Raf Proto-Oncogene) and ERBB2 (v-ErbA-Related Receptor Tyrosine Kinase 2) are also top targets with high structural druggability scores. They're ranked third and fourth, respectively, due to their moderate literature evidence (1,875 and 857 publications). BRAF is a proto-oncogene that can drive tumorigenesis when mutated, while ERBB2 is an oncogene involved in lung cancer development.
* CASP8 (Caspase 8) is the fifth-ranked target with a high structural druggability score. Although it has fewer literature evidence publications (48), its role in apoptosis regulation makes it an important target for lung cancer research.

These targets suggest that lung cancer may involve aberrant signaling pathways, such as EGFR and ERBB2-mediated activation of downstream effectors, as well as mutations in tumor suppressor genes like TP53. The involvement of BRAF and CASP8 implies a complex interplay between cell survival and apoptosis regulation.

### 🎯 Top Druggable Targets

- **EGFR** (P00533)

- **TP53** (P04637)

- **BRAF** (P15056)

---


████████████████████████████████████████████████████████████
  DONE
████████████████████████████████████████████████████████████


{'query': 'What are targets for lung cancer?',
 'corrected_query': 'What are targets for lung cancer?',
 'intent': 'DISEASE_TO_TARGET',
 'route': 'kg',
 'entities': {'Disease': ['lung cancer'],
  'Drug': [],
  'Protein': [],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [{'disease_id': 'C0242379',
    'disease_name': 'Lung cancer',
    'match_score': 54.54545454545454,
    'confidence': 'medium'}],
  'Protein': [],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 10,
 'kg_output': {'query': 'What are targets for lung cancer?',
  'intent': 'DISEASE_TO_TARGET',
  'disease': {'disease_id': 'C0242379',
   'disease_name': 'Lung cancer',
   'match_score': 54.54545454545454,
   'confidence': 'medium'},
  'druggable_targets': [{'protein_id': 'P00533',
    'gene_name': 'EGFR',
    'protein_name': 'Epidermal growth factor receptor'},
   {'protein_id': 'P04637',
    'gene_name': 'TP53',
    'protein_name': 'Cellular tumor antigen p53'},
   {'protein_id': 'P15056',
    

In [10]:
run("Which all pathways involve in Lung cancer?")


████████████████████████████████████████████████████████████
  BIOMEDICAL AGENT  |  Query: Which all pathways involve in Lung cancer?
████████████████████████████████████████████████████████████
   🔄 Natural query normalized:
      Input:  Which all pathways involve in Lung cancer?
      Output: Which pathways are involved in lung cancer?
Raw entity output: {
"Disease": ["lung cancer"],
"Drug": [],
"Protein": [],
"Molecule": [],
"Pathway": []
}
 Raw optimization output: { "limit": 10 }

🔀 Router | Intent: DISEASE_TO_PATHWAY | Route: kg
   Corrected: Which pathways are involved in lung cancer?
   Entities:  {'Disease': ['lung cancer'], 'Drug': [], 'Protein': [], 'Molecule': [], 'Pathway': []}

══════════════════════════════════════════════════
  RUNNING KG AGENT
══════════════════════════════════════════════════

🧬 KG Query | Intent: DISEASE_TO_PATHWAY

🧪 Pathways involved in Lung cancer:

1. Spry regulation of FGF signaling
2. Frs2-mediated activation
3. ARMS-mediated activation
4. Si

# 🧬 Biomedical Agent Results

**Query:** Which pathways are involved in lung cancer?

**Intent:** `DISEASE_TO_PATHWAY` | **Route:** `kg`

---

## 📊 Knowledge Graph Analysis


🧪 Pathways involved in Lung cancer:

1. Spry regulation of FGF signaling
2. Frs2-mediated activation
3. ARMS-mediated activation
4. Signalling to p38 via RIT and RIN
5. RAF activation
6. MAP2K and MAPK activation
7. Negative feedback regulation of MAPK pathway
8. Negative regulation of MAPK pathway
9. Signaling by moderate kinase activity BRAF mutants
10. Signaling by high-kinase activity BRAF mutants


### 🤖 LLM Explanation

Based on the computational biology results, lung cancer appears to be associated with several key signaling pathways. The most prominent ones include:

* 'RAF activation' and 'MAP2K and MAPK activation', which are ranked higher due to their direct involvement in tumor progression and proliferation.
* 'Signaling by moderate kinase activity BRAF mutants' is also a significant pathway, as it suggests that specific mutations in the BRAF gene contribute to lung cancer development.

The 'Spry regulation of FGF signaling' and 'Frs2-mediated activation' pathways are less prominent but still relevant, possibly indicating secondary or compensatory mechanisms.

Notably, there is no direct involvement of 'ARMS-mediated activation', 'Signalling to p38 via RIT and RIN', or the negative feedback regulations in lung cancer development. This suggests that these pathways may not be primary drivers of tumorigenesis in this disease.

Overall, the results highlight the importance of RAF-MAPK signaling in lung cancer, with specific BRAF mutations playing a key role.

---


████████████████████████████████████████████████████████████
  DONE
████████████████████████████████████████████████████████████


{'query': 'Which pathways are involved in lung cancer?',
 'corrected_query': 'Which pathways are involved in lung cancer?',
 'intent': 'DISEASE_TO_PATHWAY',
 'route': 'kg',
 'entities': {'Disease': ['lung cancer'],
  'Drug': [],
  'Protein': [],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [{'disease_id': 'C0242379',
    'disease_name': 'Lung cancer',
    'match_score': 54.54545454545454,
    'confidence': 'medium'}],
  'Protein': [],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 10,
 'kg_output': {'query': 'Which pathways are involved in lung cancer?',
  'intent': 'DISEASE_TO_PATHWAY',
  'disease': {'disease_id': 'C0242379',
   'disease_name': 'Lung cancer',
   'match_score': 54.54545454545454,
   'confidence': 'medium'},
  'druggable_targets': [],
  'kg_summary': '\n🧪 Pathways involved in Lung cancer:\n\n1. Spry regulation of FGF signaling\n2. Frs2-mediated activation\n3. ARMS-mediated activation\n4. Signalling to p38 via RIT and RIN\n5. RAF activation

In [ ]:
run("Which pathways involve the protein EGFR?")

In [ ]:
run("Which pathways does target P07471 participate in?")

In [ ]:
run("What are the all targets for Type 2 diabetes mellitus disease?")

In [ ]:
run("Which pathways are shared between lung cancer and breast cancer?")

In [ ]:
run("Which pathways are the biggest hubs connecting the most diseases?")

## Structure Queries

In [21]:
run("Find binding pockets for P07471")


████████████████████████████████████████████████████████████
  BIOMEDICAL AGENT  |  Query: Find binding pockets for P07471
████████████████████████████████████████████████████████████
   🔄 Natural query normalized:
      Input:  Find binding pockets for P07471
      Output: Find binding pockets for P07471
 Raw optimization output: { "limit": 10 }

🔀 Router | Intent: TARGET_POCKET | Route: structure
   Corrected: Find binding pockets for P07471
   Entities:  {'Disease': [], 'Drug': [], 'Protein': ['P07471'], 'Molecule': [], 'Pathway': []}

══════════════════════════════════════════════════
  RUNNING STRUCTURE AGENT (meeko_env)
══════════════════════════════════════════════════
   UniProt: P07471 | Protein: P07471

📋 Task written: {'uniprot_id': 'P07471', 'protein_name': 'P07471'}
   Launching meeko_env subprocess...
   STDOUT:
 icates that the pocket is strongly hydrophobic, which favors binding of the ligand.

The ligand's aromatic rings and rotatable bonds allow for π-stacking and va

# 🧬 Biomedical Agent Results

**Query:** Find binding pockets for P07471

**Intent:** `TARGET_POCKET` | **Route:** `structure`

---

## 🔬 Structural Analysis

**Protein:** P07471 | **PDB:** 1V54

**Pockets:** 112 | **Ligands:** 50


**pocket1** ← `CHEMBL470039` (score: -11.18)

Based on the provided physicochemical data, the binding mechanism is driven by a combination of hydrophobic and hydrogen bond interactions.

The dominant residue types in the pocket, such as GLY, ALA, VAL, LEU, and ILE, contribute to the strong hydrophobic ratio (0.55), which facilitates the binding


**pocket2** ← `CHEMBL470039` (score: -11.29)

Based on the provided physicochemical data, the binding mechanism is driven by hydrophobic interactions between the pocket's dominant residue types ('LEU', 'THR', 'ILE', 'VAL', and 'HIS') and the ligand's non-polar regions. The high hydrophobic ratio (0.55) suggests a strong affinity for non-polar i


**pocket3** ← `CHEMBL262462` (score: -10.52)

Based on the provided physicochemical data, the binding mechanism is driven by a combination of hydrophobic and hydrogen bond interactions.

The dominant residue types in the pocket, including ILE, HIS, THR, VAL, and LEU, are well-suited to interact with the ligand's hydrophobic regions. The high hy


████████████████████████████████████████████████████████████
  DONE
████████████████████████████████████████████████████████████


{'query': 'Find binding pockets for P07471',
 'corrected_query': 'Find binding pockets for P07471',
 'intent': 'TARGET_POCKET',
 'route': 'structure',
 'entities': {'Disease': [],
  'Drug': [],
  'Protein': ['P07471'],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [],
  'Protein': [{'protein_id': 'P07471',
    'protein_name': 'Cytochrome c oxidase subunit 6A2, mitochondrial',
    'gene_name': 'COX6A2',
    'match_score': 100,
    'confidence': 'high'}],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 10,
 'kg_output': {},
 'structure_output': {'uniprot_id': 'P07471',
  'pdb_id': '1V54',
  'pockets_detected': 112,
  'ligands_retrieved': 50,
  'docking_done': True,
  'pocket_explanations': [{'pocket': 'pocket1',
    'chembl_id': 'CHEMBL470039',
    'docking_score': -11.18,
    'explanation_snippet': 'Based on the provided physicochemical data, the binding mechanism is driven by a combination of hydrophobic and hydrogen bond interactions.\n\nThe dominant resid

In [ ]:
run("Find drug candidates for EGFR")

In [ ]:
run("Perform docking for protein P07471")

## Full Pipeline

In [ ]:
run("Identify druggable targets in Alzheimer's disease")

In [ ]:
run("Run full drug discovery pipeline for lung cancer")